In [ ]:
# Chris Williams
# Dr. Li Lei
# Created 7/22/2026
# this is my colab for the data splitter function for the LG-CAFN model
# effectively the most important model of the paper


In [ ]:
# work flow
# 1. Define the canonical 166/19/47 patient split.
# 2. Build train, validation, and test manifests from the ZIP.
# 3. Remove the six erroneous img9Se test assignments.
# 4. Add automatic leakage, label, missing-file, and shape validation.
# 5. Implement the PyTorch dataset and preprocessing exactly as specified in the manuscript.
# 6. Only then implement LG-CAFN.

In [ ]:
from google.colab import drive
from pathlib import Path
import os

def mount_drive_safely():
    standard_mount = Path("/content/drive")

    # 1. Use the standard location if it is already a real mount.
    if os.path.ismount(standard_mount):
        print("Google Drive is already mounted.")
        return standard_mount / "MyDrive"

    # 2. Try mounting to the standard location. If it fails due to existing files,
    #    then proceed to alternative locations.
    try:
        standard_mount.mkdir(parents=True, exist_ok=True)
        drive.mount(str(standard_mount))
        print(f"Google Drive mounted at: {standard_mount}")
        return standard_mount / "MyDrive"
    except ValueError as e:
        if "Mountpoint must not already contain files" in str(e):
            print(
                f"Warning: {standard_mount} contains local files and is not "
                "a Google Drive mount. Trying alternative mount points..."
            )
        else:
            raise e # Re-raise other ValueErrors

    # 3. If standard mount fails, try alternative locations.
    alternatives = [
        Path("/content/gdrive"),
        Path("/content/google_drive"),
        Path("/content/drive_mounted"),
    ]

    for mountpoint in alternatives:
        if not mountpoint.exists() or not any(mountpoint.iterdir()):
            mountpoint.mkdir(parents=True, exist_ok=True)
            try:
                drive.mount(str(mountpoint))
                print(f"Google Drive mounted at: {mountpoint}")
                return mountpoint / "MyDrive"
            except Exception as e:
                print(f"Could not mount at {mountpoint}: {e}")

    raise RuntimeError(
        "No clean Google Drive mount location is available."
    )


DRIVE_ROOT = mount_drive_safely()

print(f"\nGoogle Drive root: {DRIVE_ROOT}")
print(f"Exists: {DRIVE_ROOT.exists()}")

if not DRIVE_ROOT.exists():
    raise RuntimeError(
        "Google Drive mounted, but MyDrive was not found." # This means MyDrive within the mounted drive doesn't exist.
    )

Mounted at /content/drive
Google Drive mounted at: /content/drive

Google Drive root: /content/drive/MyDrive
Exists: True


In [ ]:
from pathlib import Path

# Search for BreaDM.zip anywhere in My Drive, starting from DRIVE_ROOT
drive_root_path = DRIVE_ROOT
matches = list(drive_root_path.rglob("BreaDM.zip"))

# Also account for alternate spellings/names that might be used for the main dataset
if not matches:
    matches = [
        p for p in drive_root_path.rglob("*.zip")
        if p.name.lower().replace(" ", "") in {
            "breadm.zip",
            "breastdm.zip",
            "breadm(1).zip", # Handle common auto-downloaded names
        }
    ]

if not matches:
    raise FileNotFoundError(
        "Could not find BreaDM.zip (or similar) anywhere in Google Drive. "
        "Please ensure the BreaDM.zip file is uploaded to your Google Drive."
    )

print("ZIP files found:")
for i, path in enumerate(matches):
    print(f"  [{i}] {path}")

# Use the first found match
BREADM_ZIP_PATH = matches[0]
print(f"\nUsing BreaDM.zip at: {BREADM_ZIP_PATH}")
print(f"Compressed size: {BREADM_ZIP_PATH.stat().st_size / 1024**3:.2f} GB")

ZIP files found:
  [0] /content/drive/MyDrive/BreaDM.zip

Using BreaDM.zip at: /content/drive/MyDrive/BreaDM.zip
Compressed size: 1.21 GB


In [ ]:
PROJECT_DIR = DRIVE_ROOT / "LG_CAFN_Reproduction"
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

SPLIT_DIR = PROJECT_DIR / "splits"
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

MANIFEST_DIR = PROJECT_DIR / "manifests"
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

DATA_DIR = PROJECT_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

STATS_DIR = PROJECT_DIR / "preprocessing_statistics"
STATS_DIR.mkdir(parents=True, exist_ok=True)

CLEANED_ZIP_PATH = DATA_DIR / "BreaDM_cleaned.zip"

print(f"Project Directory: {PROJECT_DIR}")
print(f"Splits Directory: {SPLIT_DIR}")
print(f"Manifests Directory: {MANIFEST_DIR}")
print(f"Data Directory: {DATA_DIR}")
print(f"Statistics Directory: {STATS_DIR}")
print(f"Cleaned ZIP Path: {CLEANED_ZIP_PATH}")

Project Directory: /content/drive/MyDrive/LG_CAFN_Reproduction
Splits Directory: /content/drive/MyDrive/LG_CAFN_Reproduction/splits
Manifests Directory: /content/drive/MyDrive/LG_CAFN_Reproduction/manifests
Data Directory: /content/drive/MyDrive/LG_CAFN_Reproduction/data
Statistics Directory: /content/drive/MyDrive/LG_CAFN_Reproduction/preprocessing_statistics
Cleaned ZIP Path: /content/drive/MyDrive/LG_CAFN_Reproduction/data/BreaDM_cleaned.zip


---

In [ ]:
from pathlib import Path
from collections import defaultdict, Counter
import zipfile

# Use the dynamically found BREADM_ZIP_PATH
zip_path = BREADM_ZIP_PATH

split_aliases = {
    "train": "train",
    "training": "train",
    "val": "val",
    "valid": "val",
    "validation": "val",
    "test": "test",
}

class_aliases = {
    "b": "Benign",
    "benign": "Benign",
    "m": "Malignant",
    "malignant": "Malignant",
}

# branch -> split -> class -> information
audit = defaultdict(
    lambda: defaultdict(
        lambda: defaultdict(
            lambda: {
                "files": 0,
                "patients": set(),
                "extensions": Counter(),
                "examples": [],
            }
        )
    )
)

unparsed = []

with zipfile.ZipFile(zip_path, "r") as archive:
    for info in archive.infolist():
        if info.is_dir():
            continue

        parts = Path(info.filename).parts
        lower = [part.lower() for part in parts]

        # Only inspect classification content
        if not lower or lower[0] != "cls":
            continue

        split_index = next(
            (i for i, part in enumerate(lower) if part in split_aliases),
            None,
        )

        if split_index is None:
            unparsed.append(info.filename)
            continue

        split = split_aliases[lower[split_index]]
        branch = "/".join(parts[:split_index])

        # Find the class directory after the split directory
        class_index = next(
            (
                i for i in range(split_index + 1, len(parts))
                if lower[i] in class_aliases
            ),
            None,
        )

        if class_index is None:
            unparsed.append(info.filename)
            continue

        label = class_aliases[lower[class_index]]

        # Expected structure: split/class/patient/file
        patient = (
            parts[class_index + 1]
            if class_index + 1 < len(parts) - 1
            else "UNKNOWN"
        )

        extension = Path(info.filename).suffix.lower() or "<none>"
        record = audit[branch][split][label]

        record["files"] += 1
        record["patients"].add(patient)
        record["extensions"][extension] += 1

        if len(record["examples"]) < 2:
            record["examples"].append(info.filename)

print("=" * 100)
print("CLASSIFICATION BRANCH AUDIT")
print("=" * 100)

for branch in sorted(audit):
    print(f"\nBRANCH: {branch}")

    all_split_patients = {}

    for split in ("train", "val", "test"):
        if split not in audit[branch]:
            continue

        print(f"  {split.upper()}")

        split_patients = set()

        for label in ("Benign", "Malignant"):
            if label not in audit[branch][split]:
                continue

            record = audit[branch][split][label]
            split_patients |= record["patients"]

            print(
                f"    {label:<10} "
                f"patients={len(record['patients']):>3}  "
                f"files={record['files']:>6}  "
                f"extensions={dict(record['extensions'])}"
            )

            for example in record["examples"]:
                print(f"      example: {example}")

        all_split_patients[split] = split_patients
        print(f"    TOTAL UNIQUE PATIENTS: {len(split_patients)}")

    # Check patient overlap within this specific branch
    print("  PATIENT-LEAKAGE CHECK:")

    comparisons = [
        ("train", "val"),
        ("train", "test"),
        ("val", "test"),
    ]

    any_leakage = False

    for first, second in comparisons:
        if first in all_split_patients and second in all_split_patients:
            overlap = (
                all_split_patients[first]
                & all_split_patients[second]
            )

            print(
                f"    {first} vs {second}: "
                f"{len(overlap)} overlapping patients"
            )

            if overlap:
                any_leakage = True
                print(f"      {sorted(overlap)[:15]}")

    if not any_leakage:
        print("    PASS: no patient overlap detected.")

print("\n" + "=" * 100)
print(f"Unparsed classification files: {len(unparsed):,}")

if unparsed:
    print("First 20 unparsed paths:")
    for name in unparsed[:20]:
        print(" ", name)


CLASSIFICATION BRANCH AUDIT

BRANCH: cls/GLCM/img17Se
  TRAIN
    Benign     patients= 61  files=   327  extensions={'.npy': 327}
      example: cls/GLCM/img17Se/train/Benign/BreaDM-Be-1801/p-032.npy
      example: cls/GLCM/img17Se/train/Benign/BreaDM-Be-1801/p-033.npy
    Malignant  patients=105  files=   875  extensions={'.npy': 875}
      example: cls/GLCM/img17Se/train/Malignant/BreaDM-Ma-1802/p-035.npy
      example: cls/GLCM/img17Se/train/Malignant/BreaDM-Ma-1802/p-036.npy
    TOTAL UNIQUE PATIENTS: 166
  VAL
    Benign     patients=  7  files=    24  extensions={'.npy': 24}
      example: cls/GLCM/img17Se/val/Benign/BreaDM-Be-1802/p-040.npy
      example: cls/GLCM/img17Se/val/Benign/BreaDM-Be-1802/p-041.npy
    Malignant  patients= 12  files=    93  extensions={'.npy': 93}
      example: cls/GLCM/img17Se/val/Malignant/BreaDM-Ma-1805/p-032.npy
      example: cls/GLCM/img17Se/val/Malignant/BreaDM-Ma-1805/p-033.npy
    TOTAL UNIQUE PATIENTS: 19
  TEST
    Benign     patients= 17  f

In [ ]:
from pathlib import Path
from collections import defaultdict, Counter
import zipfile

# Use the dynamically found BREADM_ZIP_PATH
zip_path = BREADM_ZIP_PATH

split_aliases = {
    "train": "train",
    "training": "train",
    "val": "val",
    "valid": "val",
    "validation": "val",
    "test": "test",
}

class_aliases = {
    "b": "Benign",
    "benign": "Benign",
    "m": "Malignant",
    "malignant": "Malignant",
}

# branch -> split -> class -> information
audit = defaultdict(
    lambda: defaultdict(
        lambda: defaultdict(
            lambda: {
                "files": 0,
                "patients": set(),
                "extensions": Counter(),
                "examples": [],
            }
        )
    )
)

unparsed = []

with zipfile.ZipFile(zip_path, "r") as archive:
    for info in archive.infolist():
        if info.is_dir():
            continue

        parts = Path(info.filename).parts
        lower = [part.lower() for part in parts]

        # Only inspect classification content
        if not lower or lower[0] != "cls":
            continue

        split_index = next(
            (i for i, part in enumerate(lower) if part in split_aliases),
            None,
        )

        if split_index is None:
            unparsed.append(info.filename)
            continue

        split = split_aliases[lower[split_index]]
        branch = "/".join(parts[:split_index])

        # Find the class directory after the split directory
        class_index = next(
            (
                i for i in range(split_index + 1, len(parts))
                if lower[i] in class_aliases
            ),
            None,
        )

        if class_index is None:
            unparsed.append(info.filename)
            continue

        label = class_aliases[lower[class_index]]

        # Expected structure: split/class/patient/file
        patient = (
            parts[class_index + 1]
            if class_index + 1 < len(parts) - 1
            else "UNKNOWN"
        )

        extension = Path(info.filename).suffix.lower() or "<none>"
        record = audit[branch][split][label]

        record["files"] += 1
        record["patients"].add(patient)
        record["extensions"][extension] += 1

        if len(record["examples"]) < 2:
            record["examples"].append(info.filename)

print("=" * 100)
print("CLASSIFICATION BRANCH AUDIT")
print("=" * 100)

for branch in sorted(audit):
    print(f"\nBRANCH: {branch}")

    all_split_patients = {}

    for split in ("train", "val", "test"):
        if split not in audit[branch]:
            continue

        print(f"  {split.upper()}")

        split_patients = set()

        for label in ("Benign", "Malignant"):
            if label not in audit[branch][split]:
                continue

            record = audit[branch][split][label]
            split_patients |= record["patients"]

            print(
                f"    {label:<10} "
                f"patients={len(record['patients']):>3}  "
                f"files={record['files']:>6}  "
                f"extensions={dict(record['extensions'])}"
            )

            for example in record["examples"]:
                print(f"      example: {example}")

        all_split_patients[split] = split_patients
        print(f"    TOTAL UNIQUE PATIENTS: {len(split_patients)}")

    # Check patient overlap within this specific branch
    print("  PATIENT-LEAKAGE CHECK:")

    comparisons = [
        ("train", "val"),
        ("train", "test"),
        ("val", "test"),
    ]

    any_leakage = False

    for first, second in comparisons:
        if first in all_split_patients and second in all_split_patients:
            overlap = (
                all_split_patients[first]
                & all_split_patients[second]
            )

            print(
                f"    {first} vs {second}: "
                f"{len(overlap)} overlapping patients"
            )

            if overlap:
                any_leakage = True
                print(f"      {sorted(overlap)[:15]}")

    if not any_leakage:
        print("    PASS: no patient overlap detected.")

print("\n" + "=" * 100)
print(f"Unparsed classification files: {len(unparsed):,}")

if unparsed:
    print("First 20 unparsed paths:")
    for name in unparsed[:20]:
        print(" ", name)


CLASSIFICATION BRANCH AUDIT

BRANCH: cls/GLCM/img17Se
  TRAIN
    Benign     patients= 61  files=   327  extensions={'.npy': 327}
      example: cls/GLCM/img17Se/train/Benign/BreaDM-Be-1801/p-032.npy
      example: cls/GLCM/img17Se/train/Benign/BreaDM-Be-1801/p-033.npy
    Malignant  patients=105  files=   875  extensions={'.npy': 875}
      example: cls/GLCM/img17Se/train/Malignant/BreaDM-Ma-1802/p-035.npy
      example: cls/GLCM/img17Se/train/Malignant/BreaDM-Ma-1802/p-036.npy
    TOTAL UNIQUE PATIENTS: 166
  VAL
    Benign     patients=  7  files=    24  extensions={'.npy': 24}
      example: cls/GLCM/img17Se/val/Benign/BreaDM-Be-1802/p-040.npy
      example: cls/GLCM/img17Se/val/Benign/BreaDM-Be-1802/p-041.npy
    Malignant  patients= 12  files=    93  extensions={'.npy': 93}
      example: cls/GLCM/img17Se/val/Malignant/BreaDM-Ma-1805/p-032.npy
      example: cls/GLCM/img17Se/val/Malignant/BreaDM-Ma-1805/p-033.npy
    TOTAL UNIQUE PATIENTS: 19
  TEST
    Benign     patients= 17  f

In [ ]:
from pathlib import Path, PurePosixPath
from collections import defaultdict
import zipfile
import json
import csv

# ------------------------------------------------------------------
# Configuration
# ------------------------------------------------------------------

# Use the dynamically found BREADM_ZIP_PATH
ZIP_PATH = CLEANED_ZIP_PATH
CANONICAL_BRANCH = ("cls", "img17Se")

OUTPUT_DIR = SPLIT_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_COUNTS = {
    "train": {
        "Benign": 61,
        "Malignant": 105,
        "total": 166,
    },
    "val": {
        "Benign": 7,
        "Malignant": 12,
        "total": 19,
    },
    "test": {
        "Benign": 17,
        "Malignant": 30,
        "total": 47,
    },
}

VALID_SPLITS = {"train", "val", "test"}
VALID_LABELS = {"Benign", "Malignant"}

# split -> label -> patient IDs
canonical_split = {
    split: {
        label: set()
        for label in VALID_LABELS
    }
    for split in VALID_SPLITS
}

# ------------------------------------------------------------------
# Derive patient assignments from cls/img17Se
#
# Expected archive structure:
# cls/img17Se/{split}/{label}/{patient_id}/{file}.npy
# ------------------------------------------------------------------

with zipfile.ZipFile(ZIP_PATH, "r") as archive:
    for info in archive.infolist():
        if info.is_dir():
            continue

        parts = PurePosixPath(info.filename).parts

        if len(parts) < 6:
            continue

        if tuple(parts[:2]) != CANONICAL_BRANCH:
            continue

        split = parts[2]
        label = parts[3]
        patient_id = parts[4]

        if split not in VALID_SPLITS:
            continue

        if label not in VALID_LABELS:
            raise ValueError(
                f"Unexpected label in canonical branch: {label}"
            )

        canonical_split[split][label].add(patient_id)

# ------------------------------------------------------------------
# Validate expected patient and class counts
# ------------------------------------------------------------------

for split in ("train", "val", "test"):
    split_total = 0

    for label in ("Benign", "Malignant"):
        actual = len(canonical_split[split][label])
        expected = EXPECTED_COUNTS[split][label]

        assert actual == expected, (
            f"{split}/{label}: expected {expected} patients, "
            f"but found {actual}."
        )

        split_total += actual

    assert split_total == EXPECTED_COUNTS[split]["total"], (
        f"{split}: expected {EXPECTED_COUNTS[split]["total"]} "
        f"patients, but found {split_total}."
    )

# ------------------------------------------------------------------
# Validate patient-level separation
# ------------------------------------------------------------------

patient_sets = {
    split: (
        canonical_split[split]["Benign"]
        | canonical_split[split]["Malignant"]
    )
    for split in ("train", "val", "test")
}

assert patient_sets["train"].isdisjoint(patient_sets["val"]), (
    "Patient leakage detected between train and validation."
)

assert patient_sets["train"].isdisjoint(patient_sets["test"]), (
    "Patient leakage detected between train and test."
)

assert patient_sets["val"].isdisjoint(patient_sets["test"]), (
    "Patient leakage detected between validation and test."
)

all_patients = set().union(*patient_sets.values())

assert len(all_patients) == 232, (
    f"Expected 232 unique patients, but found {len(all_patients)}."
)

# Verify that no patient appears with both labels
benign_patients = set().union(
    *[
        canonical_split[split]["Benign"]
        for split in ("train", "val", "test")
    ]
)

malignant_patients = set().union(
    *[
        canonical_split[split]["Malignant"]
        for split in ("train", "val", "test")
    ]
)

assert benign_patients.isdisjoint(malignant_patients), (
    "At least one patient appears under both class labels."
)

# ------------------------------------------------------------------
# Convert sets to sorted lists for reproducible serialization
# ------------------------------------------------------------------

canonical_split_serializable = {
    split: {
        label: sorted(canonical_split[split][label])
        for label in ("Benign", "Malignant")
    }
    for split in ("train", "val", "test")
}

# Convenient patient-to-assignment lookup
patient_assignments = {}

for split in ("train", "val", "test"):
    for label in ("Benign", "Malignant"):
        for patient_id in canonical_split_serializable[split][label]:
            patient_assignments[patient_id] = {
                "split": split,
                "label": label,
                "label_index": 0 if label == "Benign" else 1,
            }

# ------------------------------------------------------------------
# Save JSON
# ------------------------------------------------------------------

json_path = OUTPUT_DIR / "canonical_patient_split.json"

json_data = {
    "source_archive": ZIP_PATH.name,
    "source_branch": "/".join(CANONICAL_BRANCH),
    "total_patients": len(all_patients),
    "label_mapping": {
        "Benign": 0,
        "Malignant": 1,
    },
    "splits": canonical_split_serializable,
}

with open(json_path, "w", encoding="utf-8") as file:
    json.dump(json_data, file, indent=2)

# ------------------------------------------------------------------
# Save flat CSV manifest
# ------------------------------------------------------------------

csv_path = OUTPUT_DIR / "canonical_patient_split.csv"

with open(csv_path, "w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(
        file,
        fieldnames=[
            "patient_id",
            "split",
            "label",
            "label_index",
        ],
    )
    writer.writeheader()

    for patient_id in sorted(patient_assignments):
        assignment = patient_assignments[patient_id]

        writer.writerow({
            "patient_id": patient_id,
            "split": assignment["split"],
            "label": assignment["label"],
            "label_index": assignment["label_index"],
        })

# ------------------------------------------------------------------
# Report
# ------------------------------------------------------------------

print("Canonical LG-CAFN patient split created successfully.\n")

for split in ("train", "val", "test"):
    benign_count = len(canonical_split[split]["Benign"])
    malignant_count = len(canonical_split[split]["Malignant"])
    total_count = len(patient_sets[split])

    print(
        f"{split:<5} | "
        f"Benign: {benign_count:>3} | "
        f"Malignant: {malignant_count:>3} | "
        f"Total: {total_count:>3}"
    )

print(f"\nUnique patients: {len(all_patients)}")
print("Patient leakage: none")
print(f"JSON saved to: {json_path}")
print(f"CSV saved to:  {csv_path}")

Canonical LG-CAFN patient split created successfully.

train | Benign:  61 | Malignant: 105 | Total: 166
val   | Benign:   7 | Malignant:  12 | Total:  19
test  | Benign:  17 | Malignant:  30 | Total:  47

Unique patients: 232
Patient leakage: none
JSON saved to: /content/drive/MyDrive/LG_CAFN_Reproduction/splits/canonical_patient_split.json
CSV saved to:  /content/drive/MyDrive/LG_CAFN_Reproduction/splits/canonical_patient_split.csv


In [ ]:
from pathlib import Path, PurePosixPath
from collections import defaultdict
import zipfile
import json
import csv

# ------------------------------------------------------------------
# Configuration
# ------------------------------------------------------------------

# Use the dynamically found BREADM_ZIP_PATH
ZIP_PATH = BREADM_ZIP_PATH
CANONICAL_BRANCH = ("cls", "img17Se")

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/LG_CAFN_Reproduction/splits"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_COUNTS = {
    "train": {
        "Benign": 61,
        "Malignant": 105,
        "total": 166,
    },
    "val": {
        "Benign": 7,
        "Malignant": 12,
        "total": 19,
    },
    "test": {
        "Benign": 17,
        "Malignant": 30,
        "total": 47,
    },
}

VALID_SPLITS = {"train", "val", "test"}
VALID_LABELS = {"Benign", "Malignant"}

# split -> label -> patient IDs
canonical_split = {
    split: {
        label: set()
        for label in VALID_LABELS
    }
    for split in VALID_SPLITS
}

# ------------------------------------------------------------------
# Derive patient assignments from cls/img17Se
#
# Expected archive structure:
# cls/img17Se/{split}/{label}/{patient_id}/{file}.npy
# ------------------------------------------------------------------

with zipfile.ZipFile(ZIP_PATH, "r") as archive:
    for info in archive.infolist():
        if info.is_dir():
            continue

        parts = PurePosixPath(info.filename).parts

        if len(parts) < 6:
            continue

        if tuple(parts[:2]) != CANONICAL_BRANCH:
            continue

        split = parts[2]
        label = parts[3]
        patient_id = parts[4]

        if split not in VALID_SPLITS:
            continue

        if label not in VALID_LABELS:
            raise ValueError(
                f"Unexpected label in canonical branch: {label}"
            )

        canonical_split[split][label].add(patient_id)

# ------------------------------------------------------------------
# Validate expected patient and class counts
# ------------------------------------------------------------------

for split in ("train", "val", "test"):
    split_total = 0

    for label in ("Benign", "Malignant"):
        actual = len(canonical_split[split][label])
        expected = EXPECTED_COUNTS[split][label]

        assert actual == expected, (
            f"{split}/{label}: expected {expected} patients, "
            f"but found {actual}."
        )

        split_total += actual

    assert split_total == EXPECTED_COUNTS[split]["total"], (
        f"{split}: expected {EXPECTED_COUNTS[split]["total"]} "
        f"patients, but found {split_total}."
    )

# ------------------------------------------------------------------
# Validate patient-level separation
# ------------------------------------------------------------------

patient_sets = {
    split: (
        canonical_split[split]["Benign"]
        | canonical_split[split]["Malignant"]
    )
    for split in ("train", "val", "test")
}

assert patient_sets["train"].isdisjoint(patient_sets["val"]), (
    "Patient leakage detected between train and validation."
)

assert patient_sets["train"].isdisjoint(patient_sets["test"]), (
    "Patient leakage detected between train and test."
)

assert patient_sets["val"].isdisjoint(patient_sets["test"]), (
    "Patient leakage detected between validation and test."
)

all_patients = set().union(*patient_sets.values())

assert len(all_patients) == 232, (
    f"Expected 232 unique patients, but found {len(all_patients)}."
)

# Verify that no patient appears with both labels
benign_patients = set().union(
    *[
        canonical_split[split]["Benign"]
        for split in ("train", "val", "test")
    ]
)

malignant_patients = set().union(
    *[
        canonical_split[split]["Malignant"]
        for split in ("train", "val", "test")
    ]
)

assert benign_patients.isdisjoint(malignant_patients), (
    "At least one patient appears under both class labels."
)

# ------------------------------------------------------------------
# Convert sets to sorted lists for reproducible serialization
# ------------------------------------------------------------------

canonical_split_serializable = {
    split: {
        label: sorted(canonical_split[split][label])
        for label in ("Benign", "Malignant")
    }
    for split in ("train", "val", "test")
}

# Convenient patient-to-assignment lookup
patient_assignments = {}

for split in ("train", "val", "test"):
    for label in ("Benign", "Malignant"):
        for patient_id in canonical_split_serializable[split][label]:
            patient_assignments[patient_id] = {
                "split": split,
                "label": label,
                "label_index": 0 if label == "Benign" else 1,
            }

# ------------------------------------------------------------------
# Save JSON
# ------------------------------------------------------------------

json_path = OUTPUT_DIR / "canonical_patient_split.json"

json_data = {
    "source_archive": ZIP_PATH.name,
    "source_branch": "/".join(CANONICAL_BRANCH),
    "total_patients": len(all_patients),
    "label_mapping": {
        "Benign": 0,
        "Malignant": 1,
    },
    "splits": canonical_split_serializable,
}

with open(json_path, "w", encoding="utf-8") as file:
    json.dump(json_data, file, indent=2)

# ------------------------------------------------------------------
# Save flat CSV manifest
# ------------------------------------------------------------------

csv_path = OUTPUT_DIR / "canonical_patient_split.csv"

with open(csv_path, "w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(
        file,
        fieldnames=[
            "patient_id",
            "split",
            "label",
            "label_index",
        ],
    )
    writer.writeheader()

    for patient_id in sorted(patient_assignments):
        assignment = patient_assignments[patient_id]

        writer.writerow({
            "patient_id": patient_id,
            "split": assignment["split"],
            "label": assignment["label"],
            "label_index": assignment["label_index"],
        })

# ------------------------------------------------------------------
# Report
# ------------------------------------------------------------------

print("Canonical LG-CAFN patient split created successfully.\n")

for split in ("train", "val", "test"):
    benign_count = len(canonical_split[split]["Benign"])
    malignant_count = len(canonical_split[split]["Malignant"])
    total_count = len(patient_sets[split])

    print(
        f"{split:<5} | "
        f"Benign: {benign_count:>3} | "
        f"Malignant: {malignant_count:>3} | "
        f"Total: {total_count:>3}"
    )

print(f"\nUnique patients: {len(all_patients)}")
print("Patient leakage: none")
print(f"JSON saved to: {json_path}")
print(f"CSV saved to:  {csv_path}")


Canonical LG-CAFN patient split created successfully.

train | Benign:  61 | Malignant: 105 | Total: 166
val   | Benign:   7 | Malignant:  12 | Total:  19
test  | Benign:  17 | Malignant:  30 | Total:  47

Unique patients: 232
Patient leakage: none
JSON saved to: /content/drive/MyDrive/LG_CAFN_Reproduction/splits/canonical_patient_split.json
CSV saved to:  /content/drive/MyDrive/LG_CAFN_Reproduction/splits/canonical_patient_split.csv


In [ ]:
from pathlib import Path, PurePosixPath
from collections import Counter, defaultdict
import zipfile
import json
import csv

# ------------------------------------------------------------------
# Configuration
# ------------------------------------------------------------------

# Use the dynamically found BREADM_ZIP_PATH
ZIP_PATH = CLEANED_ZIP_PATH

SPLIT_JSON = SPLIT_DIR / "canonical_patient_split.json"

MANIFEST_DIR_VAR = MANIFEST_DIR
MANIFEST_DIR_VAR.mkdir(parents=True, exist_ok=True)

BRANCHES = {
    "img9Se": ("cls", "img9Se"),
    "img17Se": ("cls", "img17Se"),
}

VALID_SPLITS = {"train", "val", "test"}
VALID_LABELS = {"Benign", "Malignant"}
VALID_EXTENSIONS = {".npy"}

LABEL_TO_INDEX = {
    "Benign": 0,
    "Malignant": 1,
}

# ------------------------------------------------------------------
# Load canonical patient assignments
# ------------------------------------------------------------------

with open(SPLIT_JSON, "r", encoding="utf-8") as file:
    split_data = json.load(file)

patient_assignments = {}

for split, labels in split_data["splits"].items():
    for label, patient_ids in labels.items():
        for patient_id in patient_ids:
            if patient_id in patient_assignments:
                raise ValueError(
                    f"Duplicate canonical assignment: {patient_id}"
                )

            patient_assignments[patient_id] = {
                "split": split,
                "label": label,
                "label_index": LABEL_TO_INDEX[label],
            }

assert len(patient_assignments) == 232, (
    f"Expected 232 canonical patients, "
    f"found {len(patient_assignments)}."
)

# ------------------------------------------------------------------
# Build manifests directly from the ZIP
# ------------------------------------------------------------------

manifest_rows = {
    branch_name: []
    for branch_name in BRANCHES
}

excluded_rows = []
unknown_patients = []
label_mismatches = []

with zipfile.ZipFile(ZIP_PATH, "r") as archive:
    for info in archive.infolist():
        if info.is_dir():
            continue

        parts = PurePosixPath(info.filename).parts
        extension = PurePosixPath(info.filename).suffix.lower()

        if extension not in VALID_EXTENSIONS:
            continue

        for branch_name, branch_prefix in BRANCHES.items():
            if tuple(parts[:2]) != branch_prefix:
                continue

            # Expected:
            # cls/{branch}/{physical_split}/{label}/
            # {patient_id}/{filename}.npy
            if len(parts) != 6:
                excluded_rows.append({
                    "branch": branch_name,
                    "archive_path": info.filename,
                    "reason": "unexpected_path_structure",
                })
                break

            physical_split = parts[2]
            physical_label = parts[3]
            patient_id = parts[4]
            filename = parts[5]

            if physical_split not in VALID_SPLITS:
                excluded_rows.append({
                    "branch": branch_name,
                    "archive_path": info.filename,
                    "reason": "invalid_physical_split",
                })
                break

            if physical_label not in VALID_LABELS:
                excluded_rows.append({
                    "branch": branch_name,
                    "archive_path": info.filename,
                    "reason": "invalid_label",
                })
                break

            if patient_id not in patient_assignments:
                unknown_patients.append({
                    "branch": branch_name,
                    "archive_path": info.filename,
                    "patient_id": patient_id,
                })
                break

            assignment = patient_assignments[patient_id]
            canonical_split = assignment["split"]
            canonical_label = assignment["label"]

            if physical_label != canonical_label:
                label_mismatches.append({
                    "branch": branch_name,
                    "archive_path": info.filename,
                    "patient_id": patient_id,
                    "physical_label": physical_label,
                    "canonical_label": canonical_label,
                })
                break

            # Exclude files physically stored in a split that does
            # not match the patient's canonical assignment.
            if physical_split != canonical_split:
                excluded_rows.append({
                    "branch": branch_name,
                    "archive_path": info.filename,
                    "patient_id": patient_id,
                    "physical_split": physical_split,
                    "canonical_split": canonical_split,
                    "reason": "noncanonical_split_assignment",
                })
                break

            manifest_rows[branch_name].append({
                "sample_id": (
                    f"{branch_name}:{patient_id}:{filename}"
                ),
                "branch": branch_name,
                "split": canonical_split,
                "label": canonical_label,
                "label_index": assignment["label_index"],
                "patient_id": patient_id,
                "filename": filename,
                "archive_path": info.filename,
                "compressed_bytes": info.compress_size,
                "uncompressed_bytes": info.file_size,
            })

            break

# ------------------------------------------------------------------
# Fail on unknown patients or label inconsistencies
# ------------------------------------------------------------------

if unknown_patients:
    examples = unknown_patients[:10]
    raise ValueError(
        f"Found {len(unknown_patients)} files belonging to "
        f"unknown patients. Examples: {examples}"
    )

if label_mismatches:
    examples = label_mismatches[:10]
    raise ValueError(
        f"Found {len(label_mismatches)} label mismatches. "
        f"Examples: {examples}"
    )

# ------------------------------------------------------------------
# Validate each branch
# ------------------------------------------------------------------

for branch_name, rows in manifest_rows.items():
    if not rows:
        raise ValueError(
            f"No samples found for branch {branch_name}."
        )

    sample_ids = [row["sample_id"] for row in rows]

    if len(sample_ids) != len(set(sample_ids)):
        duplicates = [
            sample_id
            for sample_id, count in Counter(sample_ids).items()
            if count > 1
        ]

        raise ValueError(
            f"{branch_name} contains duplicate sample IDs: "
            f"{duplicates[:10]}"
        )

    patients_by_split = defaultdict(set)

    for row in rows:
        patients_by_split[row["split"]].add(
            row["patient_id"]
        )

    assert patients_by_split["train"].isdisjoint(
        patients_by_split["val"]
    ), f"{branch_name}: train/validation patient leakage."

    assert patients_by_split["train"].isdisjoint(
        patients_by_split["test"]
    ), f"{branch_name}: train/test patient leakage."

    assert patients_by_split["val"].isdisjoint(
        patients_by_split["test"]
    ), f"{branch_name}: validation/test patient leakage."

    observed_patients = set().union(
        *patients_by_split.values()
    )

    assert observed_patients == set(patient_assignments), (
        f"{branch_name}: manifest patient set does not match "
        "the canonical 232-patient set."
    )

# ------------------------------------------------------------------
# Write manifests
# ------------------------------------------------------------------

FIELDNAMES = [
    "sample_id",
    "branch",
    "split",
    "label",
    "label_index",
    "patient_id",
    "filename",
    "archive_path",
    "compressed_bytes",
    "uncompressed_bytes",
]

for branch_name, rows in manifest_rows.items():
    rows = sorted(
        rows,
        key=lambda row: (
            row["split"],
            row["label_index"],
            row["patient_id"],
            row["filename"],
        ),
    )

    # Combined manifest for this branch
    combined_path = (
        MANIFEST_DIR_VAR / f"{branch_name}_manifest.csv"
    )

    with open(
        combined_path,
        "w",
        newline="",
        encoding="utf-8",
    ) as file:
        writer = csv.DictWriter(
            file,
            fieldnames=FIELDNAMES,
        )
        writer.writeheader()
        writer.writerows(rows)

    # Separate train, validation, and test manifests
    for split in ("train", "val", "test"):
        split_rows = [
            row for row in rows
            if row["split"] == split
        ]

        split_path = (
            MANIFEST_DIR_VAR
            / f"{branch_name}_{split}_manifest.csv"
        )

        with open(
            split_path,
            "w",
            newline="",
            encoding="utf-8",
        ) as file:
            writer = csv.DictWriter(
                file,
                fieldnames=FIELDNAMES,
            )
            writer.writeheader()
            writer.writerows(split_rows)

# Save exclusions for transparency
EXCLUSION_FIELDS = sorted({
    key
    for row in excluded_rows
    for key in row
})

exclusion_path = MANIFEST_DIR_VAR / "excluded_files.csv"

with open(
    exclusion_path,
    "w",
    newline="",
    encoding="utf-8",
) as file:
    writer = csv.DictWriter(
        file,
        fieldnames=EXCLUSION_FIELDS,
    )
    writer.writeheader()

    for row in excluded_rows:
        writer.writerow(row)

# ------------------------------------------------------------------
# Summary
# ------------------------------------------------------------------

print("MANIFEST BUILD COMPLETE")
print("=" * 80)

for branch_name, rows in manifest_rows.items():
    print(f"\n{branch_name}")

    for split in ("train", "val", "test"):
        split_rows = [
            row for row in rows
            if row["split"] == split
        ]

        patients = {
            row["patient_id"]
            for row in split_rows
        }

        benign_files = sum(
            row["label"] == "Benign"
            for row in split_rows
        )

        malignant_files = sum(
            row["label"] == "Malignant"
            for row in split_rows
        )

        print(
            f"  {split:<5} | "
            f"patients={len(patients):>3} | "
            f"samples={len(split_rows):>4} | "
            f"benign={benign_files:>3} | "
            f"malignant={malignant_files:>3}"
        )

print("\nExcluded files by reason:")
for reason, count in Counter(
    row["reason"] for row in excluded_rows
).items():
    print(f"  {reason}: {count}")

print(f"\nManifests saved to: {MANIFEST_DIR_VAR}")
print(f"Exclusion report:   {exclusion_path}")
print("Patient leakage:    none")

MANIFEST BUILD COMPLETE

img9Se
  train | patients=166 | samples=1202 | benign=327 | malignant=875
  val   | patients= 19 | samples= 117 | benign= 24 | malignant= 93
  test  | patients= 47 | samples= 403 | benign=114 | malignant=289

img17Se
  train | patients=166 | samples=1202 | benign=327 | malignant=875
  val   | patients= 19 | samples= 117 | benign= 24 | malignant= 93
  test  | patients= 47 | samples= 403 | benign=114 | malignant=289

Excluded files by reason:

Manifests saved to: /content/drive/MyDrive/LG_CAFN_Reproduction/manifests
Exclusion report:   /content/drive/MyDrive/LG_CAFN_Reproduction/manifests/excluded_files.csv
Patient leakage:    none


In [ ]:
# build train validation and test manifests from the zip
from pathlib import Path, PurePosixPath
from collections import Counter, defaultdict
import zipfile
import json
import csv

# ------------------------------------------------------------------
# Configuration
# ------------------------------------------------------------------

# Use the dynamically found BREADM_ZIP_PATH
ZIP_PATH = BREADM_ZIP_PATH

SPLIT_JSON = Path(
    "/content/drive/MyDrive/LG_CAFN_Reproduction/splits"
    "/canonical_patient_split.json"
)

MANIFEST_DIR = Path(
    "/content/drive/MyDrive/LG_CAFN_Reproduction/manifests"
)
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

BRANCHES = {
    "img9Se": ("cls", "img9Se"),
    "img17Se": ("cls", "img17Se"),
}

VALID_SPLITS = {"train", "val", "test"}
VALID_LABELS = {"Benign", "Malignant"}
VALID_EXTENSIONS = {".npy"}

LABEL_TO_INDEX = {
    "Benign": 0,
    "Malignant": 1,
}

# ------------------------------------------------------------------
# Load canonical patient assignments
# ------------------------------------------------------------------

with open(SPLIT_JSON, "r", encoding="utf-8") as file:
    split_data = json.load(file)

patient_assignments = {}

for split, labels in split_data["splits"].items():
    for label, patient_ids in labels.items():
        for patient_id in patient_ids:
            if patient_id in patient_assignments:
                raise ValueError(
                    f"Duplicate canonical assignment: {patient_id}"
                )

            patient_assignments[patient_id] = {
                "split": split,
                "label": label,
                "label_index": LABEL_TO_INDEX[label],
            }

assert len(patient_assignments) == 232, (
    f"Expected 232 canonical patients, "
    f"found {len(patient_assignments)}."
)

# ------------------------------------------------------------------
# Build manifests directly from the ZIP
# ------------------------------------------------------------------

manifest_rows = {
    branch_name: []
    for branch_name in BRANCHES
}

excluded_rows = []
unknown_patients = []
label_mismatches = []

with zipfile.ZipFile(ZIP_PATH, "r") as archive:
    for info in archive.infolist():
        if info.is_dir():
            continue

        parts = PurePosixPath(info.filename).parts
        extension = PurePosixPath(info.filename).suffix.lower()

        if extension not in VALID_EXTENSIONS:
            continue

        for branch_name, branch_prefix in BRANCHES.items():
            if tuple(parts[:2]) != branch_prefix:
                continue

            # Expected:
            # cls/{branch}/{physical_split}/{label}/
            # {patient_id}/{filename}.npy
            if len(parts) != 6:
                excluded_rows.append({
                    "branch": branch_name,
                    "archive_path": info.filename,
                    "reason": "unexpected_path_structure",
                })
                break

            physical_split = parts[2]
            physical_label = parts[3]
            patient_id = parts[4]
            filename = parts[5]

            if physical_split not in VALID_SPLITS:
                excluded_rows.append({
                    "branch": branch_name,
                    "archive_path": info.filename,
                    "reason": "invalid_physical_split",
                })
                break

            if physical_label not in VALID_LABELS:
                excluded_rows.append({
                    "branch": branch_name,
                    "archive_path": info.filename,
                    "reason": "invalid_label",
                })
                break

            if patient_id not in patient_assignments:
                unknown_patients.append({
                    "branch": branch_name,
                    "archive_path": info.filename,
                    "patient_id": patient_id,
                })
                break

            assignment = patient_assignments[patient_id]
            canonical_split = assignment["split"]
            canonical_label = assignment["label"]

            if physical_label != canonical_label:
                label_mismatches.append({
                    "branch": branch_name,
                    "archive_path": info.filename,
                    "patient_id": patient_id,
                    "physical_label": physical_label,
                    "canonical_label": canonical_label,
                })
                break

            # Exclude files physically stored in a split that does
            # not match the patient's canonical assignment.
            if physical_split != canonical_split:
                excluded_rows.append({
                    "branch": branch_name,
                    "archive_path": info.filename,
                    "patient_id": patient_id,
                    "physical_split": physical_split,
                    "canonical_split": canonical_split,
                    "reason": "noncanonical_split_assignment",
                })
                break

            manifest_rows[branch_name].append({
                "sample_id": (
                    f"{branch_name}:{patient_id}:{filename}"
                ),
                "branch": branch_name,
                "split": canonical_split,
                "label": canonical_label,
                "label_index": assignment["label_index"],
                "patient_id": patient_id,
                "filename": filename,
                "archive_path": info.filename,
                "compressed_bytes": info.compress_size,
                "uncompressed_bytes": info.file_size,
            })

            break

# ------------------------------------------------------------------
# Fail on unknown patients or label inconsistencies
# ------------------------------------------------------------------

if unknown_patients:
    examples = unknown_patients[:10]
    raise ValueError(
        f"Found {len(unknown_patients)} files belonging to "
        f"unknown patients. Examples: {examples}"
    )

if label_mismatches:
    examples = label_mismatches[:10]
    raise ValueError(
        f"Found {len(label_mismatches)} label mismatches. "
        f"Examples: {examples}"
    )

# ------------------------------------------------------------------
# Validate each branch
# ------------------------------------------------------------------

for branch_name, rows in manifest_rows.items():
    if not rows:
        raise ValueError(
            f"No samples found for branch {branch_name}."
        )

    sample_ids = [row["sample_id"] for row in rows]

    if len(sample_ids) != len(set(sample_ids)):
        duplicates = [
            sample_id
            for sample_id, count in Counter(sample_ids).items()
            if count > 1
        ]

        raise ValueError(
            f"{branch_name} contains duplicate sample IDs: "
            f"{duplicates[:10]}"
        )

    patients_by_split = defaultdict(set)

    for row in rows:
        patients_by_split[row["split"]].add(
            row["patient_id"]
        )

    assert patients_by_split["train"].isdisjoint(
        patients_by_split["val"]
    ), f"{branch_name}: train/validation patient leakage."

    assert patients_by_split["train"].isdisjoint(
        patients_by_split["test"]
    ), f"{branch_name}: train/test patient leakage."

    assert patients_by_split["val"].isdisjoint(
        patients_by_split["test"]
    ), f"{branch_name}: validation/test patient leakage."

    observed_patients = set().union(
        *patients_by_split.values()
    )

    assert observed_patients == set(patient_assignments), (
        f"{branch_name}: manifest patient set does not match "
        "the canonical 232-patient set."
    )

# ------------------------------------------------------------------
# Write manifests
# ------------------------------------------------------------------

FIELDNAMES = [
    "sample_id",
    "branch",
    "split",
    "label",
    "label_index",
    "patient_id",
    "filename",
    "archive_path",
    "compressed_bytes",
    "uncompressed_bytes",
]

for branch_name, rows in manifest_rows.items():
    rows = sorted(
        rows,
        key=lambda row: (
            row["split"],
            row["label_index"],
            row["patient_id"],
            row["filename"],
        ),
    )

    # Combined manifest for this branch
    combined_path = (
        MANIFEST_DIR / f"{branch_name}_manifest.csv"
    )

    with open(
        combined_path,
        "w",
        newline="",
        encoding="utf-8",
    ) as file:
        writer = csv.DictWriter(
            file,
            fieldnames=FIELDNAMES,
        )
        writer.writeheader()
        writer.writerows(rows)

    # Separate train, validation, and test manifests
    for split in ("train", "val", "test"):
        split_rows = [
            row for row in rows
            if row["split"] == split
        ]

        split_path = (
            MANIFEST_DIR
            / f"{branch_name}_{split}_manifest.csv"
        )

        with open(
            split_path,
            "w",
            newline="",
            encoding="utf-8",
        ) as file:
            writer = csv.DictWriter(
                file,
                fieldnames=FIELDNAMES,
            )
            writer.writeheader()
            writer.writerows(split_rows)

# Save exclusions for transparency
EXCLUSION_FIELDS = sorted({
    key
    for row in excluded_rows
    for key in row
})

exclusion_path = MANIFEST_DIR / "excluded_files.csv"

with open(
    exclusion_path,
    "w",
    newline="",
    encoding="utf-8",
) as file:
    writer = csv.DictWriter(
        file,
        fieldnames=EXCLUSION_FIELDS,
    )
    writer.writeheader()

    for row in excluded_rows:
        writer.writerow(row)

# ------------------------------------------------------------------
# Summary
# ------------------------------------------------------------------

print("MANIFEST BUILD COMPLETE")
print("=" * 80)

for branch_name, rows in manifest_rows.items():
    print(f"\n{branch_name}")

    for split in ("train", "val", "test"):
        split_rows = [
            row for row in rows
            if row["split"] == split
        ]

        patients = {
            row["patient_id"]
            for row in split_rows
        }

        benign_files = sum(
            row["label"] == "Benign"
            for row in split_rows
        )

        malignant_files = sum(
            row["label"] == "Malignant"
            for row in split_rows
        )

        print(
            f"  {split:<5} | "
            f"patients={len(patients):>3} | "
            f"samples={len(split_rows):>4} | "
            f"benign={benign_files:>3} | "
            f"malignant={malignant_files:>3}"
        )

print("\nExcluded files by reason:")
for reason, count in Counter(
    row["reason"] for row in excluded_rows
).items():
    print(f"  {reason}: {count}")

print(f"\nManifests saved to: {MANIFEST_DIR}")
print(f"Exclusion report:   {exclusion_path}")
print("Patient leakage:    none")


MANIFEST BUILD COMPLETE

img9Se
  train | patients=166 | samples=1202 | benign=327 | malignant=875
  val   | patients= 19 | samples= 117 | benign= 24 | malignant= 93
  test  | patients= 47 | samples= 403 | benign=114 | malignant=289

img17Se
  train | patients=166 | samples=1202 | benign=327 | malignant=875
  val   | patients= 19 | samples= 117 | benign= 24 | malignant= 93
  test  | patients= 47 | samples= 403 | benign=114 | malignant=289

Excluded files by reason:
  noncanonical_split_assignment: 43

Manifests saved to: /content/drive/MyDrive/LG_CAFN_Reproduction/manifests
Exclusion report:   /content/drive/MyDrive/LG_CAFN_Reproduction/manifests/excluded_files.csv
Patient leakage:    none


In [ ]:
from pathlib import Path, PurePosixPath
from collections import defaultdict
import zipfile
import shutil
import hashlib

# ------------------------------------------------------------------
# Configuration
# ------------------------------------------------------------------

# Use the dynamically found BREADM_ZIP_PATH
SOURCE_ZIP = BREADM_ZIP_PATH
TEMP_CLEAN_ZIP = Path("/content/BreaDM_cleaned.zip")
DRIVE_CLEAN_ZIP = CLEANED_ZIP_PATH

DRIVE_CLEAN_ZIP.parent.mkdir(parents=True, exist_ok=True)

ERRONEOUS_PATIENTS = {
    "BreaDM-Ma-1802",
    "BreaDM-Ma-1803",
    "BreaDM-Ma-1804",
    "BreaDM-Ma-1806",
    "BreaDM-Ma-1807",
    "BreaDM-Ma-1808",
}

EXPECTED_REMOVED_FILES = 43

# ------------------------------------------------------------------
# Safety checks
# ------------------------------------------------------------------

if not SOURCE_ZIP.exists():
    raise FileNotFoundError(f"Source ZIP not found: {SOURCE_ZIP}")

source_size = SOURCE_ZIP.stat().st_size
free_space = shutil.disk_usage("/content").free

# Allow room for a cleaned copy and temporary ZIP overhead
required_space = int(source_size * 1.20)

if free_space < required_space:
    raise RuntimeError(
        "Insufficient Colab disk space.\n"
        f"Required: approximately {required_space / 1024**3:.2f} GB\n"
        f"Available: {free_space / 1024**3:.2f} GB"
    )

if TEMP_CLEAN_ZIP.exists():
    TEMP_CLEAN_ZIP.unlink()

# The previous error occurred here because the file existed and we want to overwrite it.
# Remove the FileExistsError check to allow overwriting.
# if DRIVE_CLEAN_ZIP.exists():
#     raise FileExistsError(
#         f"A cleaned archive already exists at:\n{DRIVE_CLEAN_ZIP}\n"
#         "Rename or remove it manually before rerunning this cell."
#     )

# ------------------------------------------------------------------
# Identify the exact files to remove
# ------------------------------------------------------------------

removed_paths = []
removed_by_patient = defaultdict(list)

with zipfile.ZipFile(SOURCE_ZIP, "r") as source:
    for info in source.infolist():
        if info.is_dir():
            continue

        parts = PurePosixPath(info.filename).parts

        # Exact expected path:
        # cls/img9Se/test/Malignant/{patient_id}/{file}.npy
        should_remove = (
            len(parts) == 6
            and parts[0] == "cls"
            and parts[1] == "img9Se"
            and parts[2] == "test"
            and parts[3] == "Malignant"
            and parts[4] in ERRONEOUS_PATIENTS
            and PurePosixPath(parts[5]).suffix.lower() == ".npy"
        )

        if should_remove:
            removed_paths.append(info.filename)
            removed_by_patient[parts[4]].append(info.filename)

# ------------------------------------------------------------------
# Validate removal target before writing anything
# ------------------------------------------------------------------

found_patients = set(removed_by_patient)

assert found_patients == ERRONEOUS_PATIENTS, (
    "The patients found in the removal target do not exactly match "
    "the six expected anomalous patients.\n"
    f"Expected: {sorted(ERRONEOUS_PATIENTS)}\n"
    f"Found: {sorted(found_patients)}"
)

assert len(removed_paths) == EXPECTED_REMOVED_FILES, (
    f"Expected exactly {EXPECTED_REMOVED_FILES} files to remove, "
    f"but found {len(removed_paths)}. No archive was modified."
)

removed_path_set = set(removed_paths)

print("Validated removal plan:")
for patient_id in sorted(removed_by_patient):
    print(
        f"  {patient_id}: "
        f"{len(removed_by_patient[patient_id])} files"
    )

print(f"\nTotal files scheduled for removal: {len(removed_paths)}")

# ------------------------------------------------------------------
# Create a new cleaned ZIP
#
# The original BreaDM.zip is opened read-only and remains unchanged.
# ------------------------------------------------------------------

copied_files = 0

with zipfile.ZipFile(SOURCE_ZIP, "r") as source:
    with zipfile.ZipFile(
        TEMP_CLEAN_ZIP,
        "w",
        compression=zipfile.ZIP_DEFLATED,
        compresslevel=6,
        allowZip64=True,
    ) as destination:

        for info in source.infolist():
            if info.filename in removed_path_set:
                continue

            # Preserve directory entries and file metadata.
            data = source.read(info.filename)

            new_info = zipfile.ZipInfo(
                filename=info.filename,
                date_time=info.date_time,
            )

            new_info.comment = info.comment
            new_info.extra = info.extra
            new_info.internal_attr = info.internal_attr
            new_info.external_attr = info.external_attr
            new_info.create_system = info.create_system
            new_info.flag_bits = info.flag_bits
            new_info.compress_type = zipfile.ZIP_DEFLATED

            destination.writestr(
                new_info,
                data,
                compress_type=zipfile.ZIP_DEFLATED,
                compresslevel=6,
            )

            if not info.is_dir():
                copied_files += 1

print("\nCleaned archive created locally.")
print(f"Files copied: {copied_files:,}")

# ------------------------------------------------------------------
# Verify the cleaned ZIP
# ------------------------------------------------------------------

remaining_erroneous_paths = []
cleaned_file_count = 0

with zipfile.ZipFile(TEMP_CLEAN_ZIP, "r") as cleaned:
    bad_member = cleaned.testzip()

    if bad_member is not None:
        raise RuntimeError(
            f"ZIP integrity test failed at: {bad_member}"
        )

    for info in cleaned.infolist():
        if info.is_dir():
            continue

        cleaned_file_count += 1
        parts = PurePosixPath(info.filename).parts

        is_erroneous_test_assignment = (
            len(parts) == 6
            and parts[0] == "cls"
            and parts[1] == "img9Se"
            and parts[2] == "test"
            and parts[3] == "Malignant"
            and parts[4] in ERRONEOUS_PATIENTS
        )

        if is_erroneous_test_assignment:
            remaining_erroneous_paths.append(info.filename)

assert not remaining_erroneous_paths, (
    "Verification failed: erroneous paths remain in the cleaned ZIP."
)

with zipfile.ZipFile(SOURCE_ZIP, "r") as source:
    original_file_count = sum(
        not info.is_dir()
        for info in source.infolist()
    )

assert cleaned_file_count == (
    original_file_count - EXPECTED_REMOVED_FILES
), (
    "The cleaned archive file count is not equal to the original "
    "file count minus 43."
)

# ------------------------------------------------------------------
# Save the cleaned archive to Google Drive
# ------------------------------------------------------------------

print("\nCopying verified archive to Google Drive...")
shutil.copy2(TEMP_CLEAN_ZIP, DRIVE_CLEAN_ZIP)

assert DRIVE_CLEAN_ZIP.exists()
assert DRIVE_CLEAN_ZIP.stat().st_size == TEMP_CLEAN_ZIP.stat().st_size

# ------------------------------------------------------------------
# Produce a SHA-256 checksum for reproducibility
# ------------------------------------------------------------------

sha256 = hashlib.sha256()

with open(DRIVE_CLEAN_ZIP, "rb") as file:
    for block in iter(lambda: file.read(8 * 1024 * 1024), b""):
        sha256.update(block)

checksum = sha256.hexdigest()

print("\nCLEANING COMPLETE")
print("=" * 80)
print(f"Original files: {original_file_count:,}")
print(f"Removed files:  {EXPECTED_REMOVED_FILES}")
print(f"Cleaned files:  {cleaned_file_count:,}")
print(f"Removed patients: {len(ERRONEOUS_PATIENTS)}")
print("ZIP integrity: PASS")
print(f"SHA-256: {checksum}")
print(f"\nCleaned archive saved to:\n{DRIVE_CLEAN_ZIP}")
print(f"\nOriginal archive preserved at:\n{SOURCE_ZIP}")

Validated removal plan:
  BreaDM-Ma-1802: 13 files
  BreaDM-Ma-1803: 9 files
  BreaDM-Ma-1804: 4 files
  BreaDM-Ma-1806: 5 files
  BreaDM-Ma-1807: 8 files
  BreaDM-Ma-1808: 4 files

Total files scheduled for removal: 43

Cleaned archive created locally.
Files copied: 70,766

Copying verified archive to Google Drive...

CLEANING COMPLETE
Original files: 70,809
Removed files:  43
Cleaned files:  70,766
Removed patients: 6
ZIP integrity: PASS
SHA-256: a4a033b0eec512bb07e859866ad93cba116524863094d2130bdb7fd48b6aa0dd

Cleaned archive saved to:
/content/drive/MyDrive/LG_CAFN_Reproduction/data/BreaDM_cleaned.zip

Original archive preserved at:
/content/drive/MyDrive/BreaDM.zip


In [ ]:
print("\n--- Verifying file persistence ---")

expected_files = [
    SPLIT_DIR / "canonical_patient_split.json",
    SPLIT_DIR / "canonical_patient_split.csv",
    MANIFEST_DIR / "img9Se_manifest.csv",
    MANIFEST_DIR / "img9Se_train_manifest.csv",
    MANIFEST_DIR / "img9Se_val_manifest.csv",
    MANIFEST_DIR / "img9Se_test_manifest.csv",
    MANIFEST_DIR / "img17Se_manifest.csv",
    MANIFEST_DIR / "img17Se_train_manifest.csv",
    MANIFEST_DIR / "img17Se_val_manifest.csv",
    MANIFEST_DIR / "img17Se_test_manifest.csv",
    MANIFEST_DIR / "excluded_files.csv",
    CLEANED_ZIP_PATH
]

all_found = True
for f_path in expected_files:
    if f_path.exists():
        print(f"[✓] Found: {f_path}")
    else:
        print(f"[✗] NOT Found: {f_path}")
        all_found = False

if all_found:
    print("\nAll required files are found at their Google Drive locations.")
else:
    print("\nWARNING: Some required files were not found.")


--- Verifying file persistence ---
[✓] Found: /content/drive/MyDrive/LG_CAFN_Reproduction/splits/canonical_patient_split.json
[✓] Found: /content/drive/MyDrive/LG_CAFN_Reproduction/splits/canonical_patient_split.csv
[✓] Found: /content/drive/MyDrive/LG_CAFN_Reproduction/manifests/img9Se_manifest.csv
[✓] Found: /content/drive/MyDrive/LG_CAFN_Reproduction/manifests/img9Se_train_manifest.csv
[✓] Found: /content/drive/MyDrive/LG_CAFN_Reproduction/manifests/img9Se_val_manifest.csv
[✓] Found: /content/drive/MyDrive/LG_CAFN_Reproduction/manifests/img9Se_test_manifest.csv
[✓] Found: /content/drive/MyDrive/LG_CAFN_Reproduction/manifests/img17Se_manifest.csv
[✓] Found: /content/drive/MyDrive/LG_CAFN_Reproduction/manifests/img17Se_train_manifest.csv
[✓] Found: /content/drive/MyDrive/LG_CAFN_Reproduction/manifests/img17Se_val_manifest.csv
[✓] Found: /content/drive/MyDrive/LG_CAFN_Reproduction/manifests/img17Se_test_manifest.csv
[✓] Found: /content/drive/MyDrive/LG_CAFN_Reproduction/manifests/excl

In [ ]:
from pathlib import Path, PurePosixPath
from collections import Counter, defaultdict
from io import BytesIO
import zipfile
import pandas as pd
import numpy as np

# ------------------------------------------------------------------
# Configuration
# ------------------------------------------------------------------

ZIP_PATH = Path(
    "/content/drive/MyDrive/LG_CAFN_Reproduction/"
    "data/BreaDM_cleaned.zip"
)

MANIFEST_DIR = Path(
    "/content/drive/MyDrive/LG_CAFN_Reproduction/manifests"
)

BRANCHES = ("img9Se", "img17Se")
SPLITS = ("train", "val", "test")

EXPECTED_PATIENT_COUNTS = {
    "train": 166,
    "val": 19,
    "test": 47,
}

EXPECTED_SAMPLE_COUNTS = {
    "train": 1202,
    "val": 117,
    "test": 403,
}

EXPECTED_CLASS_PATIENT_COUNTS = {
    "train": {"Benign": 61, "Malignant": 105},
    "val": {"Benign": 7, "Malignant": 12},
    "test": {"Benign": 17, "Malignant": 30},
}

EXPECTED_CLASS_SAMPLE_COUNTS = {
    "train": {"Benign": 327, "Malignant": 875},
    "val": {"Benign": 24, "Malignant": 93},
    "test": {"Benign": 114, "Malignant": 289},
}

LABEL_TO_INDEX = {
    "Benign": 0,
    "Malignant": 1,
}

PATIENT_PREFIX_TO_LABEL = {
    "BreaDM-Be-": "Benign",
    "BreaDM-Ma-": "Malignant",
}

REQUIRED_COLUMNS = {
    "sample_id",
    "branch",
    "split",
    "label",
    "label_index",
    "patient_id",
    "filename",
    "archive_path",
}

# Adding expected channels for shape validation
EXPECTED_CHANNELS = {
    "img9Se": 9,
    "img17Se": 17,
}

# ------------------------------------------------------------------
# Error collection
# ------------------------------------------------------------------

errors = []
warnings = []

def record_error(message):
    errors.append(message)

def infer_label_from_patient_id(patient_id):
    patient_id = str(patient_id)

    for prefix, label in PATIENT_PREFIX_TO_LABEL.items():
        if patient_id.startswith(prefix):
            return label

    return None

# ------------------------------------------------------------------
# Initial file checks
# ------------------------------------------------------------------

if not ZIP_PATH.exists():
    raise FileNotFoundError(f"Cleaned ZIP not found: {ZIP_PATH}")

for branch in BRANCHES:
    manifest_path = MANIFEST_DIR / f"{branch}_manifest.csv"

    if not manifest_path.exists():
        raise FileNotFoundError(
            f"Manifest not found: {manifest_path}"
        )

print("Opening cleaned archive and reading manifests...")

# ------------------------------------------------------------------
# Read ZIP directory once
# ------------------------------------------------------------------

with zipfile.ZipFile(ZIP_PATH, "r") as archive:
    bad_zip_member = archive.testzip()

    if bad_zip_member is not None:
        raise RuntimeError(
            f"ZIP integrity failure at: {bad_zip_member}"
        )

    zip_file_paths = {
        info.filename
        for info in archive.infolist()
        if not info.is_dir()
    }

    # Keep only model-input arrays in the two relevant branches.
    zip_branch_paths = {
        branch: {
            name
            for name in zip_file_paths
            if name.startswith(f"cls/{branch}/")
            and name.lower().endswith(".npy")
        }
        for branch in BRANCHES
    }

    manifest_frames = {}

    # ==============================================================
    # Manifest-level validation
    # ==============================================================

    for branch in BRANCHES:
        print(f"\nValidating manifest structure: {branch}")

        manifest_path = (
            MANIFEST_DIR / f"{branch}_manifest.csv"
        )

        frame = pd.read_csv(
            manifest_path,
            dtype={
                "sample_id": str,
                "branch": str,
                "split": str,
                "label": str,
                "patient_id": str,
                "filename": str,
                "archive_path": str,
            },
        )

        manifest_frames[branch] = frame

        # Required columns
        missing_columns = REQUIRED_COLUMNS - set(frame.columns)

        if missing_columns:
            record_error(
                f"{branch}: missing manifest columns "
                f"{sorted(missing_columns)}"
            )
            continue

        # Missing values
        null_counts = (
            frame[list(REQUIRED_COLUMNS)]
            .isna()
            .sum()
        )

        for column, count in null_counts.items():
            if count:
                record_error(
                    f"{branch}: column '{column}' contains "
                    f"{count} missing values."
                )

        # Duplicate manifest records
        duplicate_sample_ids = frame[
            frame["sample_id"].duplicated(keep=False)
        ]

        if not duplicate_sample_ids.empty:
            record_error(
                f"{branch}: found "
                f"{len(duplicate_sample_ids)} rows with "
                "duplicate sample IDs."
            )

        duplicate_archive_paths = frame[
            frame["archive_path"].duplicated(keep=False)
        ]

        if not duplicate_archive_paths.empty:
            record_error(
                f"{branch}: found "
                f"{len(duplicate_archive_paths)} rows with "
                "duplicate archive paths."
            )

        # Branch and split names
        invalid_branches = set(frame["branch"]) - {branch}

        if invalid_branches:
            record_error(
                f"{branch}: invalid branch values "
                f"{sorted(invalid_branches)}"
            )

        invalid_splits = set(frame["split"]) - set(SPLITS)

        if invalid_splits:
            record_error(
                f"{branch}: invalid split values "
                f"{sorted(invalid_splits)}"
            )

        invalid_labels = (
            set(frame["label"]) - set(LABEL_TO_INDEX)
        )

        if invalid_labels:
            record_error(
                f"{branch}: invalid labels "
                f"{sorted(invalid_labels)}"
            )

        # ----------------------------------------------------------
        # Label validation
        # ----------------------------------------------------------

        for row in frame.itertuples(index=False):
            expected_index = LABEL_TO_INDEX.get(row.label)

            try:
                actual_index = int(row.label_index)
            except (TypeError, ValueError):
                record_error(
                    f"{branch}: invalid label_index for "
                    f"{row.archive_path}: {row.label_index}"
                )
                continue

            if expected_index != actual_index:
                record_error(
                    f"{branch}: label/index mismatch at "
                    f"{row.archive_path}: label={row.label}, "
                    f"label_index={actual_index}"
                )

            patient_label = infer_label_from_patient_id(
                row.patient_id
            )

            if patient_label is None:
                record_error(
                    f"{branch}: unrecognized patient ID format: "
                    f"{row.patient_id}"
                )
            elif patient_label != row.label:
                record_error(
                    f"{branch}: patient ID label mismatch for "
                    f"{row.patient_id}: manifest={row.label}, "
                    f"patient ID implies={patient_label}"
                )

            parts = PurePosixPath(row.archive_path).parts

            if len(parts) != 6:
                record_error(
                    f"{branch}: malformed archive path: "
                    f"{row.archive_path}"
                )
                continue

            path_branch = parts[1]
            path_split = parts[2]
            path_label = parts[3]
            path_patient = parts[4]
            path_filename = parts[5]

            comparisons = {
                "branch": (path_branch, row.branch),
                "split": (path_split, row.split),
                "label": (path_label, row.label),
                "patient": (path_patient, row.patient_id),
                "filename": (path_filename, row.filename),
            }

            for field, (path_value, manifest_value) in (
                comparisons.items()
            ):
                if str(path_value) != str(manifest_value):
                    record_error(
                        f"{branch}: {field} mismatch at "
                        f"{row.archive_path}: path={path_value}, "
                        f"manifest={manifest_value}"
                    )

        # ----------------------------------------------------------
        # Missing and unmanifested ZIP files
        # ----------------------------------------------------------

        manifest_paths = set(frame["archive_path"])
        expected_zip_paths = zip_branch_paths[branch]

        missing_files = manifest_paths - zip_file_paths
        unmanifested_files = expected_zip_paths - manifest_paths

        if missing_files:
            record_error(
                f"{branch}: {len(missing_files)} manifest files "
                "are missing from the cleaned ZIP. Examples: "
                f"{sorted(missing_files)[:10]}"
            )

        if unmanifested_files:
            record_error(
                f"{branch}: {len(unmanifested_files)} ZIP files "
                "are absent from the manifest. Examples: "
                f"{sorted(unmanifested_files)[:10]}"
            )

        # ----------------------------------------------------------
        # Patient leakage
        # ----------------------------------------------------------

        patients_by_split = {
            split: set(
                frame.loc[
                    frame["split"] == split,
                    "patient_id",
                ]
            )
            for split in SPLITS
        }

        for first, second in (
            ("train", "val"),
            ("train", "test"),
            ("val", "test"),
        ):
            overlap = (
                patients_by_split[first]
                & patients_by_split[second]
            )

            if overlap:
                record_error(
                    f"{branch}: patient leakage between "
                    f"{first} and {second}: "
                    f"{sorted(overlap)}"
                )

        # Sample-level leakage, independent of archive path
        sample_keys_by_split = {}

        for split in SPLITS:
            split_frame = frame[frame["split"] == split]

            sample_keys_by_split[split] = {
                (row.patient_id, row.filename)
                for row in split_frame.itertuples(index=False)
            }

        for first, second in (
            ("train", "val"),
            ("train", "test"),
            ("val", "test"),
        ):
            overlap = (
                sample_keys_by_split[first]
                & sample_keys_by_split[second]
            )

            if overlap:
                record_error(
                    f"{branch}: sample leakage between "
                    f"{first} and {second}: "
                    f"{sorted(overlap)[:10]}"
                )

        # ----------------------------------------------------------
        # Expected count validation
        # ----------------------------------------------------------

        for split in SPLITS:
            split_frame = frame[
                frame["split"] == split
            ]

            actual_samples = len(split_frame)
            actual_patients = (
                split_frame["patient_id"].nunique()
            )

            if (
                actual_samples
                != EXPECTED_SAMPLE_COUNTS[split]
            ):
                record_error(
                    f"{branch}/{split}: expected "
                    f"{EXPECTED_SAMPLE_COUNTS[split]} samples, "
                    f"found {actual_samples}."
                )

            if (
                actual_patients
                != EXPECTED_PATIENT_COUNTS[split]
            ):
                record_error(
                    f"{branch}/{split}: expected "
                    f"{EXPECTED_PATIENT_COUNTS[split]} patients, "
                    f"found {actual_patients}."
                )

            for label in LABEL_TO_INDEX:
                label_frame = split_frame[
                    split_frame["label"] == label
                ]

                actual_class_samples = len(label_frame)
                actual_class_patients = (
                    label_frame["patient_id"].nunique()
                )

                expected_class_samples = (
                    EXPECTED_CLASS_SAMPLE_COUNTS
                    [split][label]
                )

                expected_class_patients = (
                    EXPECTED_CLASS_PATIENT_COUNTS
                    [split][label]
                )

                if (
                    actual_class_samples
                    != expected_class_samples
                ):
                    record_error(
                        f"{branch}/{split}/{label}: expected "
                        f"{expected_class_samples} samples, "
                        f"found {actual_class_samples}."
                    )

                if (
                    actual_class_patients
                    != expected_class_patients
                ):
                    record_error(
                        f"{branch}/{split}/{label}: expected "
                        f"{expected_class_patients} patients, "
                        f"found {actual_class_patients}."
                    )

    # ==============================================================
    # Array shape and readability validation (modified logic)
    # ==============================================================

    shape_counts = {}
    dtype_counts = {}
    corrupt_arrays = defaultdict(list)

    for branch in BRANCHES:
        frame = manifest_frames[branch]
        branch_shapes = Counter()
        branch_dtypes = Counter()
        expected_channels = EXPECTED_CHANNELS[branch]

        print(
            f"\nInspecting {len(frame):,} arrays in {branch}..."
        )

        for position, row in enumerate(
            frame.itertuples(index=False),
            start=1,
        ):
            try:
                raw_bytes = archive.read(row.archive_path)

                array = np.load(
                    BytesIO(raw_bytes),
                    allow_pickle=False,
                )

                branch_shapes[tuple(array.shape)] += 1
                branch_dtypes[str(array.dtype)] += 1

                # Validate individual array properties
                if array.ndim != 3:
                    record_error(
                        f"{branch}: expected a 3D H×W×C array, "
                        f"found {array.shape} at {row.archive_path}"
                    )
                    continue

                height, width, channels = array.shape

                if height <= 0 or width <= 0:
                    record_error(
                        f"{branch}: invalid spatial dimensions "
                        f"{array.shape} at {row.archive_path}"
                    )

                if channels != expected_channels:
                    record_error(
                        f"{branch}: expected {expected_channels} "
                        f"channels, found {channels} at "
                        f"{row.archive_path}"
                    )

                if array.dtype != np.uint8:
                    record_error(
                        f"{branch}: expected uint8, found "
                        f"{array.dtype} at {row.archive_path}"
                    )

                if not np.isfinite(array).all():
                    record_error(
                        f"{branch}: NaN or infinite value found at "
                        f"{row.archive_path}"
                    )

            except Exception as exception:
                corrupt_arrays[branch].append(
                    (row.archive_path, repr(exception))
                )

            if position % 250 == 0:
                print(
                    f"  Checked {position:,}/{len(frame):,}"
                )

        shape_counts[branch] = branch_shapes
        dtype_counts[branch] = branch_dtypes

        if corrupt_arrays[branch]:
            record_error(
                f"{branch}: {len(corrupt_arrays[branch])} "
                "unreadable or invalid arrays. Examples: "
                f"{corrupt_arrays[branch][:10]}"
            )

        if len(branch_shapes) == 0:
            record_error(
                f"{branch}: no readable arrays were found."
            )

        # Removed the 'elif len(branch_shapes) > 1' check here,
        # as variable spatial shapes are expected for ROIs.
        # Individual property checks are performed in the loop.


# ------------------------------------------------------------------
# Final report
# ------------------------------------------------------------------

print("\n" + "=" * 80)
print("LG-CAFN DATA VALIDATION REPORT")
print("=" * 80)

for branch in BRANCHES:
    frame = manifest_frames[branch]

    print(f"\n{branch}")

    for split in SPLITS:
        split_frame = frame[frame["split"] == split]

        print(
            f"  {split:<5} | "
            f"patients={split_frame['patient_id'].nunique():>3} | "
            f"samples={len(split_frame):>4}"
        )

    print(f"  shapes: {dict(shape_counts[branch])}")
    print(f"  dtypes: {dict(dtype_counts[branch])}")

print("\nChecks performed:")
print("  ZIP integrity")
print("  Required manifest columns")
print("  Missing manifest values")
print("  Duplicate sample IDs and archive paths")
print("  Patient-level and sample-level leakage")
print("  Label, label-index, patient-ID, and path agreement")
print("  Missing ZIP files")
print("  Unmanifested ZIP files")
print("  Expected patient, sample, and class counts")
print("  NumPy readability, numeric dtype, and shape consistency")

if warnings:
    print("\nWARNINGS:")
    for warning in warnings:
        print(f"  - {warning}")

if errors:
    print(f"\nVALIDATION FAILED: {len(errors)} issue(s) found")

    for number, error in enumerate(errors, start=1):
        print(f"  {number}. {error}")

    raise AssertionError(
        "LG-CAFN dataset validation failed. "
        "Review the errors printed above."
    )

print("\nVALIDATION PASSED")
print("No leakage, label errors, missing files, or shape errors found.")

Opening cleaned archive and reading manifests...


In [16]:
# respond to the assertion issue
from pathlib import Path
from io import BytesIO
from collections import Counter
import zipfile
import pandas as pd
import numpy as np

ZIP_PATH = Path(
    "/content/drive/MyDrive/LG_CAFN_Reproduction/"
    "data/BreaDM_cleaned.zip"
)

MANIFEST_DIR = Path(
    "/content/drive/MyDrive/LG_CAFN_Reproduction/manifests"
)

EXPECTED_CHANNELS = {
    "img9Se": 9,
    "img17Se": 17,
}

shape_records = {}
errors = []

with zipfile.ZipFile(ZIP_PATH, "r") as archive:
    for branch, expected_channels in EXPECTED_CHANNELS.items():
        manifest = pd.read_csv(
            MANIFEST_DIR / f"{branch}_manifest.csv"
        )

        branch_shapes = Counter()
        branch_records = {}

        print(f"Validating {branch}...")

        for number, row in enumerate(
            manifest.itertuples(index=False),
            start=1,
        ):
            try:
                array = np.load(
                    BytesIO(archive.read(row.archive_path)),
                    allow_pickle=False,
                )
            except Exception as exception:
                errors.append(
                    f"{branch}: unable to read "
                    f"{row.archive_path}: {exception}"
                )
                continue

            shape = tuple(array.shape)
            branch_shapes[shape] += 1

            # A stable cross-branch key for the same ROI sample
            sample_key = (
                str(row.patient_id),
                str(row.filename),
                str(row.split),
                str(row.label),
            )

            branch_records[sample_key] = {
                "shape": shape,
                "dtype": str(array.dtype),
                "archive_path": row.archive_path,
            }

            # Arrays must be H x W x C
            if array.ndim != 3:
                errors.append(
                    f"{branch}: expected a 3D H×W×C array, "
                    f"found {shape} at {row.archive_path}"
                )
                continue

            height, width, channels = shape

            if height <= 0 or width <= 0:
                errors.append(
                    f"{branch}: invalid spatial dimensions "
                    f"{shape} at {row.archive_path}"
                )

            if channels != expected_channels:
                errors.append(
                    f"{branch}: expected {expected_channels} "
                    f"channels, found {channels} at "
                    f"{row.archive_path}"
                )

            if array.dtype != np.uint8:
                errors.append(
                    f"{branch}: expected uint8, found "
                    f"{array.dtype} at {row.archive_path}"
                )

            if not np.isfinite(array).all():
                errors.append(
                    f"{branch}: NaN or infinite value found at "
                    f"{row.archive_path}"
                )

            if number % 250 == 0:
                print(
                    f"  Checked {number:,}/{len(manifest):,}"
                )

        shape_records[branch] = branch_records

        spatial_shapes = Counter(
            shape[:2]
            for shape in branch_shapes
        )

        print(f"  Arrays: {len(manifest):,}")
        print(f"  Unique spatial shapes: {len(spatial_shapes)}")
        print(
            f"  Height range: "
            f"{min(h for h, w in spatial_shapes)}–"
            f"{max(h for h, w in spatial_shapes)}"
        )
        print(
            f"  Width range: "
            f"{min(w for h, w in spatial_shapes)}–"
            f"{max(w for h, w in spatial_shapes)}"
        )
        print(f"  Required channels: {expected_channels}")
        print()

# ------------------------------------------------------------------
# Cross-branch correspondence
# ------------------------------------------------------------------

img9_keys = set(shape_records["img9Se"])
img17_keys = set(shape_records["img17Se"])

missing_from_img9 = img17_keys - img9_keys
missing_from_img17 = img9_keys - img17_keys

if missing_from_img9:
    errors.append(
        f"{len(missing_from_img9)} samples exist in img17Se "
        "but not img9Se."
    )

if missing_from_img17:
    errors.append(
        f"{len(missing_from_img17)} samples exist in img9Se "
        "but not img17Se."
    )

shared_keys = img9_keys & img17_keys
spatial_mismatches = []

for key in sorted(shared_keys):
    shape9 = shape_records["img9Se"][key]["shape"]
    shape17 = shape_records["img17Se"][key]["shape"]

    if shape9[:2] != shape17[:2]:
        spatial_mismatches.append({
            "sample": key,
            "img9Se_shape": shape9,
            "img17Se_shape": shape17,
        })

if spatial_mismatches:
    errors.append(
        f"{len(spatial_mismatches)} corresponding samples have "
        "different spatial dimensions. Examples: "
        f"{spatial_mismatches[:10]}"
    )

# ------------------------------------------------------------------
# Final result
# ------------------------------------------------------------------

print("=" * 80)
print("CORRECTED SHAPE VALIDATION REPORT")
print("=" * 80)
print(f"Matched cross-branch samples: {len(shared_keys):,}")
print(
    "Cross-branch spatial mismatches: "
    f"{len(spatial_mismatches)}"
)

if errors:
    print(f"\nVALIDATION FAILED: {len(errors)} issue(s)")

    for number, error in enumerate(errors, start=1):
        print(f"{number}. {error}")

    raise AssertionError(
        "Corrected shape validation failed."
    )

print("\nVALIDATION PASSED")
print("All arrays are valid variable-sized H×W×C tumor ROIs.")
print("img9Se consistently contains 9 channels.")
print("img17Se consistently contains 17 channels.")
print("Corresponding samples have matching spatial dimensions.")

Validating img9Se...
  Checked 250/1,722
  Checked 500/1,722
  Checked 750/1,722
  Checked 1,000/1,722
  Checked 1,250/1,722
  Checked 1,500/1,722
  Arrays: 1,722
  Unique spatial shapes: 180
  Height range: 6–75
  Width range: 9–58
  Required channels: 9

Validating img17Se...
  Checked 250/1,722
  Checked 500/1,722
  Checked 750/1,722
  Checked 1,000/1,722
  Checked 1,250/1,722
  Checked 1,500/1,722
  Arrays: 1,722
  Unique spatial shapes: 180
  Height range: 6–75
  Width range: 9–58
  Required channels: 17

CORRECTED SHAPE VALIDATION REPORT
Matched cross-branch samples: 1,722
Cross-branch spatial mismatches: 0

VALIDATION PASSED
All arrays are valid variable-sized H×W×C tumor ROIs.
img9Se consistently contains 9 channels.
img17Se consistently contains 17 channels.
Corresponding samples have matching spatial dimensions.


In [17]:
# Implement the PyTorch dataset and preprocessing exactly as specified in the Manuscript
# ok so this is an issue becuase the paper is unreproducable, which is explained in the github

# ================================================================
# LG-CAFN FAITHFUL RECONSTRUCTION:
# PyTorch Dataset, Preprocessing, DataLoaders, and Channel Adapter
#
# Verified author behavior:
#   - Resize to 256 × 256
#   - Random crop to 224 × 224
#   - Random horizontal flip
#   - Random vertical flip
#   - Batch size 32
#   - Training shuffle=True and drop_last=True
#
# Reconstruction decisions:
#   - Load H × W × 9 and H × W × 17 NumPy arrays
#   - Preserve all temporal channels
#   - Apply identical spatial transforms to every channel
#   - Use deterministic center crop for validation/testing
#   - Calculate channel statistics from training data only
#   - Map 9/17 channels to 3 with a learnable 1 × 1 adapter
# ================================================================

from pathlib import Path
from io import BytesIO
import zipfile
import json
import random

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


# ----------------------------------------------------------------
# Configuration
# ----------------------------------------------------------------

ZIP_PATH = CLEANED_ZIP_PATH

MANIFEST_DIR = MANIFEST_DIR

STATS_DIR = STATS_DIR
STATS_DIR.mkdir(parents=True, exist_ok=True)

BRANCH_CHANNELS = {
    "img9Se": 9,
    "img17Se": 17,
}

LABEL_TO_INDEX = {
    "Benign": 0,
    "Malignant": 1,
}

RESIZE_SIZE = 256
CROP_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 2
SEED = 8

HORIZONTAL_FLIP_PROBABILITY = 0.5
VERTICAL_FLIP_PROBABILITY = 0.5

# Prevent division by zero during normalization.
NORMALIZATION_EPSILON = 1e-6


# ----------------------------------------------------------------
# Reproducibility
# ----------------------------------------------------------------

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    # These settings favor reproducibility over maximum speed.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def seed_worker(worker_id):
    """Give each DataLoader worker a reproducible seed."""

    worker_seed = torch.initial_seed() % (2**32)

    np.random.seed(worker_seed)
    random.seed(worker_seed)


seed_everything()


# ----------------------------------------------------------------
# Training-only normalization statistics
# ----------------------------------------------------------------

def calculate_training_channel_statistics(
    zip_path,
    manifest_path,
    expected_channels,
):
    """
    Calculate one mean and standard deviation per temporal channel.

    Only training samples are used, which prevents information from
    the validation or test partitions entering preprocessing.

    Statistics are calculated over the original ROI pixels after
    scaling uint8 values to [0,1]. Spatial resizing is not used for
    this calculation so that interpolation does not alter the
    underlying intensity distribution.
    """

    manifest = pd.read_csv(manifest_path)
    training = manifest[manifest["split"] == "train"].copy()

    if training.empty:
        raise ValueError(
            f"No training samples found in {manifest_path}"
        )

    channel_sum = np.zeros(expected_channels, dtype=np.float64)
    channel_squared_sum = np.zeros(
        expected_channels,
        dtype=np.float64,
    )
    channel_pixel_count = np.zeros(
        expected_channels,
        dtype=np.int64,
    )

    with zipfile.ZipFile(zip_path, "r") as archive:
        for number, row in enumerate(
            training.itertuples(index=False),
            start=1,
        ):
            array = np.load(
                BytesIO(archive.read(row.archive_path)),
                allow_pickle=False,
            )

            if array.ndim != 3:
                raise ValueError(
                    f"Expected H×W×C at {row.archive_path}, "
                    f"found {array.shape}"
                )

            if array.shape[-1] != expected_channels:
                raise ValueError(
                    f"Expected {expected_channels} channels at "
                    f"{row.archive_path}, found {array.shape[-1]}"
                )

            # Scale exactly once from uint8 [0,255] to [0,1].
            array = array.astype(np.float64) / 255.0

            pixels_per_channel = array.shape[0] * array.shape[1]

            channel_sum += array.sum(axis=(0, 1))
            channel_squared_sum += np.square(array).sum(
                axis=(0, 1)
            )
            channel_pixel_count += pixels_per_channel

            if number % 250 == 0:
                print(
                    f"  Statistics: checked "
                    f"{number:,}/{len(training):,}"
                )

    means = channel_sum / channel_pixel_count

    variances = (
        channel_squared_sum / channel_pixel_count
        - np.square(means)
    )

    # Protect against tiny negative values from floating-point error.
    variances = np.maximum(variances, 0.0)
    standard_deviations = np.sqrt(variances)

    if np.any(standard_deviations < NORMALIZATION_EPSILON):
        bad_channels = np.where(
            standard_deviations < NORMALIZATION_EPSILON
        )[0].tolist()

        raise ValueError(
            "Near-zero standard deviation in channels: "
            f"{bad_channels}"
        )

    return {
        "mean": means.tolist(),
        "std": standard_deviations.tolist(),
        "training_samples": int(len(training)),
        "channel_pixel_counts": channel_pixel_count.tolist(),
        "source_manifest": str(manifest_path),
        "source_archive": str(zip_path),
        "intensity_scaling": "uint8 divided by 255",
        "statistics_partition": "train only",
    }


def load_or_calculate_statistics(branch):
    """Load saved statistics or calculate and save them once."""

    expected_channels = BRANCH_CHANNELS[branch]

    manifest_path = (
        MANIFEST_DIR / f"{branch}_manifest.csv"
    )

    statistics_path = (
        STATS_DIR / f"{branch}_training_channel_stats.json"
    )

    if statistics_path.exists():
        print(f"Loading existing statistics: {statistics_path}")

        with open(
            statistics_path,
            "r",
            encoding="utf-8",
        ) as file:
            statistics = json.load(file)

    else:
        print(f"Calculating training-only statistics: {branch}")

        statistics = calculate_training_channel_statistics(
            zip_path=ZIP_PATH,
            manifest_path=manifest_path,
            expected_channels=expected_channels,
        )

        with open(
            statistics_path,
            "w",
            encoding="utf-8",
        ) as file:
            json.dump(statistics, file, indent=2)

        print(f"Saved statistics: {statistics_path}")

    if len(statistics["mean"]) != expected_channels:
        raise ValueError(
            f"{branch}: statistics contain "
            f"{len(statistics['mean'])} means instead of "
            f"{expected_channels}."
        )

    if len(statistics["std"]) != expected_channels:
        raise ValueError(
            f"{branch}: statistics contain "
            f"{len(statistics['std'])} standard deviations "
            f"instead of {expected_channels}."
        )

    return statistics


# ----------------------------------------------------------------
# Synchronized multichannel preprocessing
# ----------------------------------------------------------------

class LGCAFNPreprocessor:
    """
    Resize, crop, flip, scale, and normalize an H×W×C ROI.

    Every geometric operation is applied to the complete tensor, so
    every temporal MRI sequence receives the same transformation.
    """

    def __init__(
        self,
        mean,
        std,
        training,
        resize_size=RESIZE_SIZE,
        crop_size=CROP_SIZE,
        horizontal_flip_probability=(
            HORIZONTAL_FLIP_PROBABILITY
        ),
        vertical_flip_probability=(
            VERTICAL_FLIP_PROBABILITY
        ),
    ):
        self.training = bool(training)
        self.resize_size = int(resize_size)
        self.crop_size = int(crop_size)

        self.horizontal_flip_probability = float(
            horizontal_flip_probability
        )
        self.vertical_flip_probability = float(
            vertical_flip_probability
        )

        # Stored as C×1×1 for channel-wise broadcasting.
        self.mean = torch.tensor(
            mean,
            dtype=torch.float32,
        ).view(-1, 1, 1)

        self.std = torch.tensor(
            std,
            dtype=torch.float32,
        ).view(-1, 1, 1)

        if self.crop_size > self.resize_size:
            raise ValueError(
                "Crop size cannot exceed resize size."
            )

        if torch.any(self.std < NORMALIZATION_EPSILON):
            raise ValueError(
                "All channel standard deviations must be positive."
            )

    def __call__(self, array):
        if not isinstance(array, np.ndarray):
            raise TypeError(
                f"Expected numpy.ndarray, found {type(array)}"
            )

        if array.ndim != 3:
            raise ValueError(
                f"Expected H×W×C, found {array.shape}"
            )

        if array.dtype != np.uint8:
            raise ValueError(
                f"Expected uint8 array, found {array.dtype}"
            )

        # H×W×C -> C×H×W and [0,255] -> [0,1]
        tensor = torch.from_numpy(
            np.ascontiguousarray(array)
        ).permute(2, 0, 1).float().div(255.0)

        if tensor.shape[0] != self.mean.shape[0]:
            raise ValueError(
                f"Array has {tensor.shape[0]} channels, but "
                f"normalization expects {self.mean.shape[0]}."
            )

        # Author code: Resize([256,256]).
        tensor = F.interpolate(
            tensor.unsqueeze(0),
            size=(self.resize_size, self.resize_size),
            mode="bilinear",
            align_corners=False,
        ).squeeze(0)

        if self.training:
            # Author code: RandomCrop(224).
            maximum_offset = (
                self.resize_size - self.crop_size
            )

            top = random.randint(0, maximum_offset)
            left = random.randint(0, maximum_offset)

            tensor = tensor[
                :,
                top : top + self.crop_size,
                left : left + self.crop_size,
            ]

            # Author code defaults to probability 0.5.
            if (
                random.random()
                < self.horizontal_flip_probability
            ):
                tensor = torch.flip(tensor, dims=(2,))

            if (
                random.random()
                < self.vertical_flip_probability
            ):
                tensor = torch.flip(tensor, dims=(1,))

        else:
            # Reconstruction decision:
            # deterministic center crop for validation/testing.
            offset = (
                self.resize_size - self.crop_size
            ) // 2

            tensor = tensor[
                :,
                offset : offset + self.crop_size,
                offset : offset + self.crop_size,
            ]

        # Per-sequence statistics calculated from training only.
        tensor = (tensor - self.mean) / self.std

        expected_shape = (
            self.mean.shape[0],
            self.crop_size,
            self.crop_size,
        )

        if tuple(tensor.shape) != expected_shape:
            raise RuntimeError(
                f"Expected output {expected_shape}, "
                f"found {tuple(tensor.shape)}"
            )

        if not torch.isfinite(tensor).all():
            raise ValueError(
                "Preprocessing generated NaN or infinite values."
            )

        return tensor.contiguous()


# ----------------------------------------------------------------
# ZIP-backed PyTorch Dataset
# ----------------------------------------------------------------

class BreastDMLGCAFNDataset(Dataset):
    """
    Load BreastDM NumPy ROIs directly from the cleaned ZIP.

    The ZIP file is opened lazily inside each DataLoader worker,
    avoiding unsafe sharing of one ZipFile object across processes.
    """

    def __init__(
        self,
        zip_path,
        manifest_path,
        split,
        branch,
        transform,
    ):
        if split not in {"train", "val", "test"}:
            raise ValueError(f"Invalid split: {split}")

        if branch not in BRANCH_CHANNELS:
            raise ValueError(f"Invalid branch: {branch}")

        self.zip_path = Path(zip_path)
        self.manifest_path = Path(manifest_path)
        self.split = split
        self.branch = branch
        self.expected_channels = BRANCH_CHANNELS[branch]
        self.transform = transform
        self._archive = None

        manifest = pd.read_csv(self.manifest_path)
        manifest = manifest[
            manifest["split"] == split
        ].copy()

        manifest = manifest.sort_values(
            by=[
                "label_index",
                "patient_id",
                "filename",
            ],
        ).reset_index(drop=True)

        if manifest.empty:
            raise ValueError(
                f"No samples found for {branch}/{split}"
            )

        if set(manifest["branch"]) != {branch}:
            raise ValueError(
                f"Manifest contains data outside {branch}."
            )

        if manifest["archive_path"].duplicated().any():
            raise ValueError(
                f"Duplicate archive paths in {branch}/{split}."
            )

        for row in manifest.itertuples(index=False):
            expected_label_index = LABEL_TO_INDEX[row.label]

            if int(row.label_index) != expected_label_index:
                raise ValueError(
                    f"Label mismatch at {row.archive_path}"
                )

        self.records = manifest.to_dict("records")

    def _get_archive(self):
        if self._archive is None:
            self._archive = zipfile.ZipFile(
                self.zip_path,
                mode="r",
            )

        return self._archive

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        record = self.records[index]
        archive = self._get_archive()

        try:
            array = np.load(
                BytesIO(
                    archive.read(record["archive_path"])
                ),
                allow_pickle=False,
            )
        except KeyError as exception:
            raise FileNotFoundError(
                f"Missing ZIP member: "
                f"{record['archive_path']}"
            ) from exception

        if array.ndim != 3:
            raise ValueError(
                f"{record['archive_path']}: "
                f"expected H×W×C, found {array.shape}"
            )

        if array.shape[-1] != self.expected_channels:
            raise ValueError(
                f"{record['archive_path']}: expected "
                f"{self.expected_channels} channels, found "
                f"{array.shape[-1]}"
            )

        image = self.transform(array)

        return {
            "image": image,
            "label": torch.tensor(
                int(record["label_index"]),
                dtype=torch.long,
            ),
            "patient_id": record["patient_id"],
            "sample_id": record["sample_id"],
            "archive_path": record["archive_path"],
            "original_shape": tuple(array.shape),
        }

    def close(self):
        if self._archive is not None:
            self._archive.close()
            self._archive = None

    def __del__(self):
        self.close()

    def __getstate__(self):
        # A worker must open its own ZIP handle.
        state = self.__dict__.copy()
        state["_archive"] = None
        return state


# ----------------------------------------------------------------
# Learnable temporal-channel adapter
# ----------------------------------------------------------------

class TemporalChannelAdapter(nn.Module):
    """
    Learn a per-pixel projection from 9 or 17 DCE-MRI channels
    into the 3 channels required by ImageNet-pretrained backbones.

    This is a documented reconstruction component, not a component
    explicitly described in the original BreastDM manuscript.
    """

    def __init__(
        self,
        input_channels,
        output_channels=3,
    ):
        super().__init__()

        if input_channels not in {9, 17}:
            raise ValueError(
                "LG-CAFN reconstruction supports 9 or 17 "
                "input channels."
            )

        self.projection = nn.Conv2d(
            in_channels=input_channels,
            out_channels=output_channels,
            kernel_size=1,
            stride=1,
            padding=0,
            bias=True,
        )

        self.normalization = nn.BatchNorm2d(
            output_channels
        )

        self.activation = nn.ReLU(inplace=True)

        self.reset_parameters()

    def reset_parameters(self):
        """
        Initialize each output as an average over all sequences.

        Small independent noise breaks symmetry between the three
        output channels while retaining a stable initial projection.
        """

        with torch.no_grad():
            base_weight = 1.0 / self.projection.in_channels

            self.projection.weight.fill_(base_weight)
            self.projection.weight.add_(
                torch.randn_like(
                    self.projection.weight
                ) * 1e-3
            )

            self.projection.bias.zero_()

    def forward(self, tensor):
        if tensor.ndim != 4:
            raise ValueError(
                "Expected batched N×C×H×W input."
            )

        if (
            tensor.shape[1]
            != self.projection.in_channels
        ):
            raise ValueError(
                f"Adapter expects "
                f"{self.projection.in_channels} channels, "
                f"found {tensor.shape[1]}."
            )

        tensor = self.projection(tensor)
        tensor = self.normalization(tensor)
        tensor = self.activation(tensor)

        return tensor


# ----------------------------------------------------------------
# DataLoader construction
# ----------------------------------------------------------------

def build_lgcafn_dataloaders(
    branch,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
):
    if branch not in BRANCH_CHANNELS:
        raise ValueError(
            f"Branch must be one of {list(BRANCH_CHANNELS)}"
        )

    manifest_path = (
        MANIFEST_DIR / f"{branch}_manifest.csv"
    )

    statistics = load_or_calculate_statistics(branch)

    train_transform = LGCAFNPreprocessor(
        mean=statistics["mean"],
        std=statistics["std"],
        training=True,
    )

    evaluation_transform = LGCAFNPreprocessor(
        mean=statistics["mean"],
        std=statistics["std"],
        training=False,
    )

    datasets = {
        "train": BreastDMLGCAFNDataset(
            zip_path=ZIP_PATH,
            manifest_path=manifest_path,
            split="train",
            branch=branch,
            transform=train_transform,
        ),
        "val": BreastDMLGCAFNDataset(
            zip_path=ZIP_PATH,
            manifest_path=manifest_path,
            split="val",
            branch=branch,
            transform=evaluation_transform,
        ),
        "test": BreastDMLGCAFNDataset(
            zip_path=ZIP_PATH,
            manifest_path=manifest_path,
            split="test",
            branch=branch,
            transform=evaluation_transform,
        ),
    }

    generator = torch.Generator()
    generator.manual_seed(SEED)

    common_arguments = {
        "batch_size": batch_size,
        "num_workers": num_workers,
        "pin_memory": torch.cuda.is_available(),
        "worker_init_fn": seed_worker,
        "generator": generator,
        "persistent_workers": num_workers > 0,
    }

    loaders = {
        "train": DataLoader(
            datasets["train"],
            shuffle=True,
            drop_last=True,
            **common_arguments,
        ),
        "val": DataLoader(
            datasets["val"],
            shuffle=False,
            drop_last=False,
            **common_arguments,
        ),
        "test": DataLoader(
            datasets["test"],
            shuffle=False,
            drop_last=False,
            **common_arguments,
        ),
    }

    adapter = TemporalChannelAdapter(
        input_channels=BRANCH_CHANNELS[branch],
        output_channels=3,
    )

    return {
        "datasets": datasets,
        "loaders": loaders,
        "adapter": adapter,
        "statistics": statistics,
    }


# ----------------------------------------------------------------
# Build and smoke-test both experiments
# ----------------------------------------------------------------

experiment_data = {}

for branch in ("img9Se", "img17Se"):
    print("\n" + "=" * 72)
    print(f"BUILDING LG-CAFN RECONSTRUCTION: {branch}")
    print("=" * 72)

    experiment = build_lgcafn_dataloaders(branch)
    experiment_data[branch] = experiment

    for split in ("train", "val", "test"):
        dataset = experiment["datasets"][split]
        loader = experiment["loaders"][split]

        print(
            f"{split:<5} | "
            f"samples={len(dataset):>4} | "
            f"batches={len(loader):>3}"
        )

    batch = next(iter(experiment["loaders"]["train"]))

    images = batch["image"]
    labels = batch["label"]

    with torch.no_grad():
        adapted_images = experiment["adapter"](images)

    expected_input_shape = (
        BATCH_SIZE,
        BRANCH_CHANNELS[branch],
        CROP_SIZE,
        CROP_SIZE,
    )

    expected_adapted_shape = (
        BATCH_SIZE,
        3,
        CROP_SIZE,
        CROP_SIZE,
    )

    assert tuple(images.shape) == expected_input_shape
    assert tuple(adapted_images.shape) == expected_adapted_shape
    assert tuple(labels.shape) == (BATCH_SIZE,)
    assert torch.isfinite(images).all()
    assert torch.isfinite(adapted_images).all()

    print(f"Raw model input:     {tuple(images.shape)}")
    print(f"Adapted CNN/ViT input: {tuple(adapted_images.shape)}")
    print(f"Labels:              {tuple(labels.shape)}")
    print("Smoke test:          PASS")

print("\n" + "=" * 72)
print("LG-CAFN RECONSTRUCTION DATA PIPELINE READY")
print("=" * 72)
print("LG-CAFN-R9:  experiment_data['img9Se']")
print("LG-CAFN-R17: experiment_data['img17Se']")


BUILDING LG-CAFN RECONSTRUCTION: img9Se
Calculating training-only statistics: img9Se
  Statistics: checked 250/1,202
  Statistics: checked 500/1,202
  Statistics: checked 750/1,202
  Statistics: checked 1,000/1,202
Saved statistics: /content/drive/MyDrive/LG_CAFN_Reproduction/preprocessing_statistics/img9Se_training_channel_stats.json
train | samples=1202 | batches= 37
val   | samples= 117 | batches=  4
test  | samples= 403 | batches= 13
Raw model input:     (32, 9, 224, 224)
Adapted CNN/ViT input: (32, 3, 224, 224)
Labels:              (32,)
Smoke test:          PASS

BUILDING LG-CAFN RECONSTRUCTION: img17Se
Calculating training-only statistics: img17Se
  Statistics: checked 250/1,202
  Statistics: checked 500/1,202
  Statistics: checked 750/1,202
  Statistics: checked 1,000/1,202
Saved statistics: /content/drive/MyDrive/LG_CAFN_Reproduction/preprocessing_statistics/img17Se_training_channel_stats.json
train | samples=1202 | batches= 37
val   | samples= 117 | batches=  4
test  | sampl

In [18]:
!pip install -q timm

import json
import math
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm

from torch.utils.data import DataLoader, Subset

In [19]:
# ================================================================
# LG-CAFN-R ARCHITECTURE
#
# Paper-specified:
#   - Local SENet50 branch
#   - Global ViT-7 branch
#   - Feature-coupling units
#   - Local/global cross-attention fusion
#   - Binary classification
#
# Reconstruction-specified:
#   - Learnable 9/17 -> 3 temporal adapter
#   - ViT-Base patch-16 embedding with first seven blocks
#   - Four bidirectional feature-coupling stages
# ================================================================


class TemporalChannelAdapter(nn.Module):
    """
    Project all 9 or 17 DCE-MRI sequences into three channels.

    This preserves every temporal sequence while allowing the use
    of ImageNet-pretrained SENet and ViT branches.
    """

    def __init__(self, input_channels):
        super().__init__()

        if input_channels not in {9, 17}:
            raise ValueError(
                "Input channels must be 9 or 17."
            )

        self.input_channels = input_channels

        self.projection = nn.Conv2d(
            input_channels,
            3,
            kernel_size=1,
            bias=True,
        )

        self.normalization = nn.BatchNorm2d(3)
        self.activation = nn.ReLU(inplace=True)

        self.reset_parameters()

    def reset_parameters(self):
        with torch.no_grad():
            # Initialize each output as the temporal mean.
            self.projection.weight.fill_(
                1.0 / self.input_channels
            )

            # Break symmetry between the three outputs.
            self.projection.weight.add_(
                1e-3
                * torch.randn_like(
                    self.projection.weight
                )
            )

            self.projection.bias.zero_()

    def forward(self, x):
        if x.ndim != 4:
            raise ValueError(
                f"Expected N×C×H×W, found {tuple(x.shape)}"
            )

        if x.shape[1] != self.input_channels:
            raise ValueError(
                f"Expected {self.input_channels} channels, "
                f"found {x.shape[1]}"
            )

        x = self.projection(x)
        x = self.normalization(x)
        x = self.activation(x)

        return x


class BidirectionalFeatureCouplingUnit(nn.Module):
    """
    Align CNN and Transformer features and exchange information
    in both directions through cross-attention.

    CNN feature:
        N × Ccnn × H × W

    Transformer tokens:
        N × (1 + number_of_patches) × Cvit
    """

    def __init__(
        self,
        cnn_channels,
        transformer_dimension=768,
        fusion_dimension=256,
        attention_heads=8,
    ):
        super().__init__()

        if fusion_dimension % attention_heads != 0:
            raise ValueError(
                "fusion_dimension must be divisible by "
                "attention_heads."
            )

        self.cnn_channels = cnn_channels
        self.transformer_dimension = transformer_dimension
        self.fusion_dimension = fusion_dimension

        self.cnn_to_fusion = nn.Conv2d(
            cnn_channels,
            fusion_dimension,
            kernel_size=1,
            bias=False,
        )

        self.token_to_fusion = nn.Linear(
            transformer_dimension,
            fusion_dimension,
            bias=False,
        )

        # CNN information updates Transformer patch tokens.
        self.cnn_to_token_attention = nn.MultiheadAttention(
            embed_dim=fusion_dimension,
            num_heads=attention_heads,
            batch_first=True,
        )

        # Transformer information updates CNN features.
        self.token_to_cnn_attention = nn.MultiheadAttention(
            embed_dim=fusion_dimension,
            num_heads=attention_heads,
            batch_first=True,
        )

        self.fusion_to_token = nn.Linear(
            fusion_dimension,
            transformer_dimension,
            bias=False,
        )

        self.fusion_to_cnn = nn.Conv2d(
            fusion_dimension,
            cnn_channels,
            kernel_size=1,
            bias=False,
        )

        self.token_norm = nn.LayerNorm(
            transformer_dimension
        )

        self.cnn_norm = nn.BatchNorm2d(
            cnn_channels
        )

        # Begin near an identity mapping for stable fine-tuning.
        self.token_scale = nn.Parameter(
            torch.tensor(1e-3)
        )
        self.cnn_scale = nn.Parameter(
            torch.tensor(1e-3)
        )

    def forward(self, cnn_feature, tokens):
        batch_size, _, height, width = cnn_feature.shape

        class_token = tokens[:, :1]
        patch_tokens = tokens[:, 1:]

        number_of_patches = patch_tokens.shape[1]
        patch_grid_size = int(math.sqrt(number_of_patches))

        if patch_grid_size**2 != number_of_patches:
            raise ValueError(
                "Transformer patch-token count must form "
                "a square spatial grid."
            )

        # --------------------------------------------------------
        # Align CNN features to the Transformer fusion dimension.
        # --------------------------------------------------------

        cnn_fusion = self.cnn_to_fusion(cnn_feature)

        cnn_sequence = cnn_fusion.flatten(2).transpose(1, 2)
        token_sequence = self.token_to_fusion(patch_tokens)

        # --------------------------------------------------------
        # CNN -> Transformer cross-attention
        # --------------------------------------------------------

        token_update, _ = self.cnn_to_token_attention(
            query=token_sequence,
            key=cnn_sequence,
            value=cnn_sequence,
            need_weights=False,
        )

        updated_patch_tokens = self.token_norm(
            patch_tokens
            + self.token_scale
            * self.fusion_to_token(token_update)
        )

        # --------------------------------------------------------
        # Transformer -> CNN cross-attention
        # --------------------------------------------------------

        updated_token_sequence = self.token_to_fusion(
            updated_patch_tokens
        )

        cnn_update, _ = self.token_to_cnn_attention(
            query=cnn_sequence,
            key=updated_token_sequence,
            value=updated_token_sequence,
            need_weights=False,
        )

        cnn_update = (
            cnn_update
            .transpose(1, 2)
            .reshape(
                batch_size,
                self.fusion_dimension,
                height,
                width,
            )
        )

        updated_cnn = self.cnn_norm(
            cnn_feature
            + self.cnn_scale
            * self.fusion_to_cnn(cnn_update)
        )

        updated_tokens = torch.cat(
            [class_token, updated_patch_tokens],
            dim=1,
        )

        return updated_cnn, updated_tokens


class LGCAFNReconstruction(nn.Module):
    """
    Faithful reconstruction of the paper's LG-CAFN concept.

    Local branch:
        ImageNet-pretrained SE-ResNet50

    Global branch:
        First seven blocks of ImageNet-pretrained ViT-Base/16

    Fusion:
        Four bidirectional feature-coupling units
    """

    def __init__(
        self,
        input_channels,
        number_of_classes=2,
        pretrained=True,
        fusion_dimension=256,
        attention_heads=8,
        dropout_probability=0.5,
    ):
        super().__init__()

        self.input_channels = input_channels
        self.number_of_classes = number_of_classes

        self.temporal_adapter = TemporalChannelAdapter(
            input_channels
        )

        # Local SENet50 branch
        self.local_branch = timm.create_model(
            "seresnet50",
            pretrained=pretrained,
            features_only=True,
            out_indices=(1, 2, 3, 4),
        )

        local_channels = (
            self.local_branch.feature_info.channels()
        )

        # Global ViT branch
        complete_vit = timm.create_model(
            "vit_base_patch16_224",
            pretrained=pretrained,
            num_classes=0,
        )

        self.patch_embed = complete_vit.patch_embed
        self.class_token = complete_vit.cls_token
        self.position_embedding = complete_vit.pos_embed
        self.position_dropout = complete_vit.pos_drop
        self.patch_dropout = complete_vit.patch_drop
        self.pre_norm = complete_vit.norm_pre

        # Paper specifies seven Transformer layers.
        self.transformer_blocks = nn.ModuleList(
            list(complete_vit.blocks[:7])
        )

        self.transformer_norm = complete_vit.norm
        self.transformer_dimension = (
            complete_vit.embed_dim
        )

        # Allocate seven blocks across four fusion stages.
        self.blocks_per_stage = (2, 2, 2, 1)

        self.coupling_units = nn.ModuleList([
            BidirectionalFeatureCouplingUnit(
                cnn_channels=channels,
                transformer_dimension=(
                    self.transformer_dimension
                ),
                fusion_dimension=fusion_dimension,
                attention_heads=attention_heads,
            )
            for channels in local_channels
        ])

        self.local_pool = nn.AdaptiveAvgPool2d(1)

        classifier_input_dimension = (
            local_channels[-1]
            + self.transformer_dimension
        )

        self.classifier = nn.Sequential(
            nn.LayerNorm(classifier_input_dimension),
            nn.Dropout(dropout_probability),
            nn.Linear(
                classifier_input_dimension,
                number_of_classes,
            ),
        )

    def create_transformer_tokens(self, image):
        patch_tokens = self.patch_embed(image)

        batch_size = patch_tokens.shape[0]

        class_token = self.class_token.expand(
            batch_size,
            -1,
            -1,
        )

        tokens = torch.cat(
            [class_token, patch_tokens],
            dim=1,
        )

        if tokens.shape[1] != self.position_embedding.shape[1]:
            raise ValueError(
                "Unexpected Transformer token count: "
                f"{tokens.shape[1]}"
            )

        tokens = tokens + self.position_embedding
        tokens = self.position_dropout(tokens)
        tokens = self.patch_dropout(tokens)
        tokens = self.pre_norm(tokens)

        return tokens

    def forward(self, x, return_features=False):
        if x.ndim != 4:
            raise ValueError(
                f"Expected N×C×H×W, found {tuple(x.shape)}"
            )

        # 9/17 temporal sequences -> 3 learned channels.
        adapted_image = self.temporal_adapter(x)

        # Extract all four SENet feature scales.
        local_features = self.local_branch(
            adapted_image
        )

        if len(local_features) != 4:
            raise RuntimeError(
                f"Expected four SENet feature maps, "
                f"found {len(local_features)}."
            )

        tokens = self.create_transformer_tokens(
            adapted_image
        )

        transformer_block_index = 0
        fused_local_features = []

        for stage_index, number_of_blocks in enumerate(
            self.blocks_per_stage
        ):
            for _ in range(number_of_blocks):
                tokens = self.transformer_blocks[
                    transformer_block_index
                ](tokens)

                transformer_block_index += 1

            fused_local, tokens = self.coupling_units[
                stage_index
            ](
                local_features[stage_index],
                tokens,
            )

            fused_local_features.append(fused_local)

        tokens = self.transformer_norm(tokens)

        local_vector = self.local_pool(
            fused_local_features[-1]
        ).flatten(1)

        global_vector = tokens[:, 0]

        fused_vector = torch.cat(
            [local_vector, global_vector],
            dim=1,
        )

        logits = self.classifier(fused_vector)

        if return_features:
            return {
                "logits": logits,
                "adapted_image": adapted_image,
                "local_vector": local_vector,
                "global_vector": global_vector,
                "fused_vector": fused_vector,
                "local_feature_shapes": [
                    tuple(feature.shape)
                    for feature in fused_local_features
                ],
                "token_shape": tuple(tokens.shape),
            }

        return logits

In [20]:
# ================================================================
# MODEL SMOKE TESTS
# ================================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(f"Device: {device}")

smoke_test_results = {}

for branch, channels in (
    ("img9Se", 9),
    ("img17Se", 17),
):
    print("\n" + "=" * 72)
    print(f"TESTING {branch}: {channels} input channels")
    print("=" * 72)

    loader = experiment_data[branch]["loaders"]["train"]
    batch = next(iter(loader))

    # Two samples are sufficient for an architectural test.
    images = batch["image"][:2].to(device)
    labels = batch["label"][:2].to(device)

    model = LGCAFNReconstruction(
        input_channels=channels,
        number_of_classes=2,
        pretrained=True,
    ).to(device)

    model.train()

    result = model(
        images,
        return_features=True,
    )

    logits = result["logits"]

    assert tuple(logits.shape) == (2, 2), (
        f"Expected logits (2,2), found "
        f"{tuple(logits.shape)}"
    )

    assert torch.isfinite(logits).all(), (
        "Non-finite logits detected."
    )

    loss = F.cross_entropy(logits, labels)

    assert torch.isfinite(loss), (
        "Non-finite loss detected."
    )

    model.zero_grad(set_to_none=True)
    loss.backward()

    gradients = [
        parameter.grad
        for parameter in model.parameters()
        if parameter.requires_grad
        and parameter.grad is not None
    ]

    assert gradients, "No gradients were produced."

    assert all(
        torch.isfinite(gradient).all()
        for gradient in gradients
    ), "A non-finite gradient was detected."

    trainable_parameters = sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )

    print(f"Input:         {tuple(images.shape)}")
    print(
        f"Adapted input: "
        f"{tuple(result['adapted_image'].shape)}"
    )
    print(
        f"Local stages:  "
        f"{result['local_feature_shapes']}"
    )
    print(f"Tokens:        {result['token_shape']}")
    print(
        f"Local vector:  "
        f"{tuple(result['local_vector'].shape)}"
    )
    print(
        f"Global vector: "
        f"{tuple(result['global_vector'].shape)}"
    )
    print(f"Logits:        {tuple(logits.shape)}")
    print(f"Loss:          {loss.item():.6f}")
    print(
        f"Trainable parameters: "
        f"{trainable_parameters:,}"
    )
    print("Forward test:  PASS")
    print("Backward test: PASS")

    smoke_test_results[branch] = {
        "input_shape": tuple(images.shape),
        "adapted_shape": tuple(
            result["adapted_image"].shape
        ),
        "logit_shape": tuple(logits.shape),
        "loss": float(loss.item()),
        "trainable_parameters": trainable_parameters,
    }

    # Release GPU memory before building the other experiment.
    del model, result, logits, loss, gradients
    torch.cuda.empty_cache()

print("\nALL LG-CAFN-R SMOKE TESTS PASSED")

Device: cpu

TESTING img9Se: 9 input channels


model.safetensors: reconstructing file:   0%|          |  0.00B /  113MB            

model.safetensors: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            

Input:         (2, 9, 224, 224)
Adapted input: (2, 3, 224, 224)
Local stages:  [(2, 256, 56, 56), (2, 512, 28, 28), (2, 1024, 14, 14), (2, 2048, 7, 7)]
Tokens:        (2, 197, 768)
Local vector:  (2, 2048)
Global vector: (2, 768)
Logits:        (2, 2)
Loss:          1.037076
Trainable parameters: 82,067,742
Forward test:  PASS
Backward test: PASS

TESTING img17Se: 17 input channels


Input:         (2, 17, 224, 224)
Adapted input: (2, 3, 224, 224)
Local stages:  [(2, 256, 56, 56), (2, 512, 28, 28), (2, 1024, 14, 14), (2, 2048, 7, 7)]
Tokens:        (2, 197, 768)
Local vector:  (2, 2048)
Global vector: (2, 768)
Logits:        (2, 2)
Loss:          0.662260
Trainable parameters: 82,067,766
Forward test:  PASS
Backward test: PASS

ALL LG-CAFN-R SMOKE TESTS PASSED


In [21]:
# ================================================================
# SINGLE OPTIMIZATION-STEP TEST
# ================================================================

branch = "img9Se"
channels = 9

loader = experiment_data[branch]["loaders"]["train"]
batch = next(iter(loader))

images = batch["image"][:2].to(device)
labels = batch["label"][:2].to(device)

model = LGCAFNReconstruction(
    input_channels=channels,
    number_of_classes=2,
    pretrained=True,
).to(device)

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.01,
    momentum=0.9,
    weight_decay=0.01,
)

model.train()
optimizer.zero_grad(set_to_none=True)

logits_before = model(images)
loss_before = F.cross_entropy(
    logits_before,
    labels,
)

loss_before.backward()

gradient_norm = torch.nn.utils.clip_grad_norm_(
    model.parameters(),
    max_norm=10.0,
)

optimizer.step()

with torch.no_grad():
    logits_after = model(images)
    loss_after = F.cross_entropy(
        logits_after,
        labels,
    )

assert torch.isfinite(loss_before)
assert torch.isfinite(loss_after)
assert torch.isfinite(gradient_norm)

print("ONE-STEP OPTIMIZATION TEST")
print("=" * 72)
print(f"Loss before step: {loss_before.item():.6f}")
print(f"Loss after step:  {loss_after.item():.6f}")
print(f"Gradient norm:    {gradient_norm.item():.6f}")
print("Optimizer:        SGD")
print("Learning rate:    0.01")
print("Momentum:         0.9")
print("Weight decay:     0.01")
print("Result:           PASS")

ONE-STEP OPTIMIZATION TEST
Loss before step: 0.990737
Loss after step:  0.523597
Gradient norm:    218.546127
Optimizer:        SGD
Learning rate:    0.01
Momentum:         0.9
Weight decay:     0.01
Result:           PASS


In [22]:
# ================================================================
# SMALL-SUBSET OVERFITTING TEST
#
# This is a diagnostic, not a reported experiment.
# The model should learn a tiny fixed subset before full training.
# ================================================================

def run_small_subset_overfit_test(
    branch="img9Se",
    subset_size=16,
    maximum_epochs=30,
    learning_rate=0.01,
):
    channels = BRANCH_CHANNELS[branch]

    full_dataset = experiment_data[
        branch
    ]["datasets"]["train"]

    subset_size = min(
        subset_size,
        len(full_dataset),
    )

    # Use fixed indices for reproducibility.
    subset_indices = list(range(subset_size))
    subset = Subset(full_dataset, subset_indices)

    generator = torch.Generator()
    generator.manual_seed(SEED)

    loader = DataLoader(
        subset,
        batch_size=min(8, subset_size),
        shuffle=True,
        drop_last=False,
        num_workers=0,
        generator=generator,
    )

    model = LGCAFNReconstruction(
        input_channels=channels,
        number_of_classes=2,
        pretrained=True,
    ).to(device)

    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=learning_rate,
        momentum=0.9,
        weight_decay=0.01,
    )

    history = []

    for epoch in range(1, maximum_epochs + 1):
        model.train()

        total_loss = 0.0
        total_correct = 0
        total_examples = 0

        for batch in loader:
            images = batch["image"].to(device)
            labels = batch["label"].to(device)

            optimizer.zero_grad(set_to_none=True)

            logits = model(images)
            loss = F.cross_entropy(logits, labels)

            if not torch.isfinite(loss):
                raise RuntimeError(
                    f"Non-finite loss at epoch {epoch}."
                )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=10.0,
            )

            optimizer.step()

            predictions = logits.argmax(dim=1)

            total_loss += (
                loss.item() * labels.size(0)
            )
            total_correct += (
                predictions == labels
            ).sum().item()
            total_examples += labels.size(0)

        epoch_loss = total_loss / total_examples
        epoch_accuracy = (
            total_correct / total_examples
        )

        history.append({
            "epoch": epoch,
            "loss": epoch_loss,
            "accuracy": epoch_accuracy,
        })

        print(
            f"Epoch {epoch:02d} | "
            f"loss={epoch_loss:.6f} | "
            f"accuracy={epoch_accuracy:.2%}"
        )

        if epoch_accuracy >= 0.95:
            print(
                "\nOVERFITTING TEST PASSED: "
                "training accuracy reached at least 95%."
            )
            break

    else:
        print(
            "\nOVERFITTING TEST DID NOT REACH 95%."
        )
        print(
            "Do not begin full training until the cause "
            "has been investigated."
        )

    return model, history


overfit_model, overfit_history = (
    run_small_subset_overfit_test(
        branch="img9Se",
        subset_size=16,
        maximum_epochs=30,
    )
)

Epoch 01 | loss=0.417015 | accuracy=75.00%
Epoch 02 | loss=0.000209 | accuracy=100.00%

OVERFITTING TEST PASSED: training accuracy reached at least 95%.
